## 1. DATA & CONFIGURE

In [ ]:
#1 Modules Load/Reload
# ── CONFIG ────────────────────────────────────────────────────────────────────
DEV_MODE = True   # True: reload modules on every run | False: simple import
# ─────────────────────────────────────────────────────────────────────────────
%matplotlib inline
import os
import warnings
import io
import contextlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

DATA_FOLDER            = "data"
OPTIONS_ARCHIVE_FOLDER = os.environ.get("OPTIONS_ARCHIVE_FOLDER", "options_archive")

if DEV_MODE:
    import importlib
    import ticker_provider      as _tp; importlib.reload(_tp)
    import data_engine          as _de; importlib.reload(_de)
    import sector_lookup        as _sl; importlib.reload(_sl)
    import regime_detector      as _rd; importlib.reload(_rd)
    import scorer_mr            as _sm; importlib.reload(_sm)
    import options_scanner      as _os; importlib.reload(_os)
    import option_move_tracker  as _om; importlib.reload(_om)
    import backtester           as _bt; importlib.reload(_bt)
    import stop_loss_forensics  as _slf; importlib.reload(_slf)

from ticker_provider     import get_tickers
from data_engine         import download_sp500_data, download_universe_data
from sector_lookup       import get_sectors
from sector_lookup       import get_sectors_and_caps
from regime_detector     import RegimeDetector
from options_scanner     import OptionsScanner
from option_move_tracker import run_option_move_backtest, summarize_option_backtest, print_option_backtest_report
from stop_loss_forensics import StopLossForensics
from contextlib          import redirect_stdout
import scorer_mr
import backtester
print(f"✅ Modules {'reloaded' if DEV_MODE else 'imported'} | DEV_MODE={'ON' if DEV_MODE else 'OFF'}")


In [ ]:
#2 Configuration cell
# Available universes (see ticker_provider.py SOURCES):
# US:     sp500, nasdaq100, dow30, russell2000
# Europe: dax40, cac40, ftse100, eurostoxx50, aex25, ftse_mib, ibex35
# Change here for a different universe -- NOT hardcoded to sp500.
UNIVERSE       = "sp500"
DATA_PERIOD    = "max"       
UNIVERSE_START = "2010-01-01"  # point-in-time membership window (survivorship-bias-free)
UNIVERSE_END   = "2026-12-31"  # -- independent of BACKTEST_START/END below, don't link them

# ── Cache policy (hours before re-downloading) ───────────────────────────────
TICKERS_CACHE_HOURS = 168   # ticker list: weekly
DATA_CACHE_HOURS    = 24    # OHLCV: daily
SECTORS_CACHE_HOURS = 24 * 30  # sector classification: monthly (rarely changes)

# ── Scanner ───────────────────────────────────────────────────────────────
TOP_N          = 60
MAX_PER_SECTOR = 30
MIN_SCORE      = 1.0

# ── MR scoring weights (must sum to 1.0) ─────────────────────────
W_RSI      = 0.30
W_BB       = 0.20
W_MR       = 0.20
W_PCR      = 0.10
W_STOCHRSI = 0.10
W_WILLIAMS = 0.10

assert abs(W_RSI + W_BB + W_MR + W_PCR + W_STOCHRSI + W_WILLIAMS - 1.0) < 1e-9, (
    "❌ MR weights do not sum to 1.0"
)

# ── Technical periods/thresholds (now REALLY adjustable, not just module defaults) ──
RSI_PERIOD       = 14
BB_PERIOD        = 20
BB_STD           = 2.0
MR_PERIOD        = 252   # 1 year, for z-score mean reversion
ATR_PERIOD       = 14
VOLUME_PERIOD    = 20
STOCH_RSI_PERIOD = 14
STOCH_SMOOTH_K   = 3
STOCH_SMOOTH_D   = 3
WILLIAMS_PERIOD  = 14

# ── Hard filters ─────────────────────────────────────────────────────────────
RSI_MAX        = 50.0     # RSI above this -> excluded
MIN_AVG_VOLUME = 500_000  # minimum average daily volume

# ── Market Cap Filter ─────────────────────────────────────────────────────────
# Tiers
#   1 = Small-cap        : $0      – $15B    (smaller names of S&P500)
#   2 = Mid-cap           : $15B    – $50B
#   3 = Large-cap          : $50B    – $150B
#   4 = Mega-cap           : $150B   – $500B
#   5 = Ultra Mega-cap     : $500B   – ∞        (bigger names of S&P500)
#   [5]        -> one tier
#   [3, 4, 5]  -> more tiers
#   None ή []  -> without filter, all universe

CAP_TIERS_SELECTED = [3, 4, 5]

# if you want your custom edges instead of defaults, you can define them here
# and then pass them as cap_tiers=CAP_TIERS_OVERRIDE in compute_scores (cell 7).

CAP_TIERS_OVERRIDE = None   # π.χ. {1:(0,10e9), 2:(10e9,40e9), 3:(40e9,120e9), 4:(120e9,400e9), 5:(400e9,float("inf"))}

In [ ]:
#3 Backtest configure
# ── Reference periods for backtest stress-testing ───────────────────────────
# Use these as start_date / end_date to test the strategy across different
# market regimes. Survivorship bias warning applies to all pre-2015 periods:
# ticker_provider.py scrapes the CURRENT S&P500 list, so companies that were
# delisted/bankrupt/acquired before today won't appear (results will look
# more favorable than reality for older crisis periods).

# ── Bear markets / crashes ───────────────────────────────────────────────────
# Global Financial Crisis (peak -> trough -> partial recovery) (not enough data to run)
#   start_date="2007-10-01", end_date="2009-12-31"
#   Peak: Oct 9 2007 (~1565) | Lehman collapse: Sep 15 2008 | Trough: Mar 9 2009 (~667, -57%)
#   ⚠️ Heavy survivorship bias (Lehman, Bear Stearns, WaMu, Wachovia, Countrywide all delisted)

# COVID crash (fastest bear market in history + V-shaped recovery)
#   start_date="2020-01-01", end_date="2020-12-31"
#   Peak: Feb 19 2020 | Trough: Mar 23 2020 (-34% in 33 days) | Recovered to highs: Aug 2020
#   Minimal survivorship bias — good clean test case

# 2022 bear market (rate-hike driven, slower grind vs. crash)
#   start_date="2022-01-01", end_date="2022-12-31"
#   Peak: Jan 3 2022 | Trough: Oct 12 2022 (-25%) | Driven by Fed tightening, not a crash

# ── Bull markets / trending ──────────────────────────────────────────────────
# Post-COVID recovery + AI rally (near-uninterrupted uptrend)
#   start_date="2023-01-01", end_date="2026-07-09"
#   Only ~1 month classified "bear" by RegimeDetector in this whole window

# 2021 melt-up (low-rate, high-liquidity bull market)
#   start_date="2021-01-01", end_date="2021-12-31"

# ── Choppy / sideways / mixed ─────────────────────────────────────────────────
# 2015-2016 (flat/choppy, oil crash, China devaluation scare, no clean trend)
#   start_date="2015-06-01", end_date="2016-06-30"

# 2018 (Q4 selloff, "almost bear" but recovered fast)
#   start_date="2018-01-01", end_date="2018-12-31"
#   Peak: Sep 20 2018 | Trough: Dec 24 2018 (-19.8%, just short of bear market) | V-shaped recovery

# ── Full-cycle stress tests ───────────────────────────────────────────────────
# Long window covering multiple regimes (bull -> bear -> bull -> bear -> bull)
#   start_date="2018-01-01", end_date="2023-12-31"

# Maximum available history (requires DATA_PERIOD="max")
#   start_date="2010-01-01", end_date=None

from membership_local import get_membership_table_local
membership = get_membership_table_local()

BACKTEST_START        = "2021-01-01"
BACKTEST_END          = "2026-06-01"   # None = until the end of the data
BACKTEST_CAPITAL      = 10_000
BACKTEST_TOP_N        = 5      # max simultaneous positions
SIGNAL_THRESHOLD      = 6.5    # min score for a new entry
EXIT_SCORE_THRESHOLD  = 5.0    # close position if score falls below this
BENCHMARK             = "SPY"
MAX_HOLD_DAYS         = 365    # safety valve
BEAR_REGIME_EXIT      = True   # close all positions if regime -> bear
USE_REGIME_FILTER     = True   # standalone RegimeDetector (VIX+SPY+breadth), no macro

# 3-phase stop loss
GUARD_DAYS         = 15    # phase 1: days without trailing
HARD_FLOOR_ATR     = 3.0   # phase 1: max loss = N×ATR
TRAIL_TRIGGER_ATR  = 1.0   # phase 3: activates after +N×ATR profit
ATR_TRAIL_MULT     = 2.5   # phase 3: trailing distance
STOP_ATR_MULT      = 1.5   # initial stop (legacy, not used in the 3-phase)
TARGET_ATR_MULT    = 4.0   # target 2 (main exit)


In [ ]:
#4 Universe & Data download
data = download_universe_data(
    index_name=UNIVERSE, start=UNIVERSE_START, end=UNIVERSE_END,
    period=DATA_PERIOD, folder_path=DATA_FOLDER, max_age_hours=DATA_CACHE_HOURS,
)
tickers = list(data.columns.get_level_values(0).unique())
print(f"Universe: {len(tickers)} tickers (point-in-time, incl. delisted)")
print(f"Data: {data.shape[0]} days, {data.columns.get_level_values(0).nunique()} tickers")


In [ ]:
#5 Missing Tickers Gap (optional diagnostic — requires DB access; install the
# "maintenance" extra of trading-shared-data and set DATABASE_URL to use this)
RUN_GAP_CHECK    = False   # True/False: run the membership_coverage_gap check
PRINT_GAP_REPORT = False   # True/False: print the report

missing_tickers = ['AABA', 'ABMD', 'ACS', 'ADS', 'AET', 'AGN', 'AKS', 'ALTR', 'ALXN', 'ANDV', 'ANRZQ', 'ANSS',
                   'APOL', 'ARG', 'ATGE', 'ATVI', 'AVP', 'AYE', 'BCR', 'BDK', 'BIG', 'BJS', 'BMS', 'BNI', 'BRCM',
                   'BTUUQ', 'BXLT', 'CA', 'CBS', 'CDAY', 'CELG', 'CEPH', 'CERN', 'CFN', 'CHK', 'CMA', 'CMCSK',
                   'COG', 'COV', 'CPGX', 'CSRA', 'CTLT', 'CTRA', 'CTXS', 'CVC', 'CVH', 'CXO', 'DAY', 'DF', 'DFS',
                   'DISCA', 'DISCK', 'DISH', 'DNB', 'DNR', 'DO', 'DRE', 'DTV', 'DWDP', 'EKDKQ', 'ENDP', 'ESRX',
                   'ESV', 'ETFC', 'FDO', 'FII', 'FL', 'FLIR', 'FRC', 'FRX', 'FTR', 'GAS', 'GGP', 'GMCR', 'GPS',
                   'HBI', 'HCBK', 'HCP', 'HES', 'HFC', 'HNZ', 'HOLX', 'HRS', 'HSH', 'HSP', 'HUBB', 'IGT', 'IPG',
                   'JCP', 'JEC', 'JNPR', 'JNS', 'JOY', 'JWN', 'K', 'KRFT', 'KSU', 'LLL', 'LLTC', 'LM', 'LO', 'LSI',
                   'LVLT', 'LXK', 'MDP', 'MFE', 'MIL', 'MJN', 'MNK', 'MON', 'MRO', 'MWV', 'MWW', 'MXIM', 'MYL',
                   'NBL', 'NLSN', 'NOVL', 'NVLS', 'NYX', 'ODP', 'PARA', 'PBCT', 'PCP', 'PDCO', 'PEAK', 'PETM',
                   'PGN', 'PLL', 'PX', 'PXD', 'QEP', 'QLGC', 'RAI', 'RDC', 'RE', 'RHT', 'RRD', 'RSHCQ', 'RTN',
                   'SCG', 'SEE', 'SIAL', 'SIVB', 'SNI', 'SRCL', 'STJ', 'STR', 'SUNEQ', 'SVU', 'SWN', 'SWY',
                   'SYMC', 'TGNA', 'TIF', 'TSS', 'TWC', 'TWTR', 'TWX', 'VAR', 'VIAB', 'VIAC', 'WBA', 'WCG',
                   'WFM', 'WIN', 'WPX', 'WRK', 'X', 'XEC', 'XL', 'XLNX', 'XTO']

result = None
if RUN_GAP_CHECK:
    from db import membership_coverage_gap, print_coverage_gap_report
    result = membership_coverage_gap(
        index_name=UNIVERSE, start=UNIVERSE_START, end=UNIVERSE_END,
        missing_tickers=missing_tickers,
    )
    if PRINT_GAP_REPORT:
        print_coverage_gap_report(result)
elif PRINT_GAP_REPORT:
    print("⚠️  RUN_GAP_CHECK=False — no result to print.")


## 2. SCORERS (MR + Options)

In [ ]:
#6 Sectors + Market Caps
sectors, market_caps = get_sectors_and_caps(
    tickers, folder_path=DATA_FOLDER, universe_name=UNIVERSE,
    max_age_hours=SECTORS_CACHE_HOURS,
)
print(f"Sectors: {len(sectors)} tickers classified")
print(f"Market caps: {sum(1 for v in market_caps.values() if v)} / {len(market_caps)} found")


In [ ]:
#7 Options Scanner 
RUN_OPTIONS_SCANNER = False  # False -> skip (faster, MR scoring without PCR)
options_df = None
if RUN_OPTIONS_SCANNER:
    active_today = set(get_tickers(UNIVERSE))
    options_tickers = [t for t in tickers if t in active_today]
    print(f"Options scan: {len(options_tickers)}/{len(tickers)} tickers (only currently active)")

    scanner = OptionsScanner()
    scanner.scan(options_tickers)
    scanner.save_full_chain_archive(folder_path=OPTIONS_ARCHIVE_FOLDER)
    scanner.print_report(min_pcr=1.0, top_n=25)
    options_df = scanner.to_dataframe()


In [ ]:
#8 Mr Scorer

_cap_tiers_cfg = CAP_TIERS_OVERRIDE or scorer_mr.CAP_TIERS

def _in_selected_tiers(mcap):
    if not CAP_TIERS_SELECTED:
        return True
    if mcap is None:
        return False
    for tier in CAP_TIERS_SELECTED:
        lo, hi = _cap_tiers_cfg.get(tier, (0, float("inf")))
        if lo <= mcap < hi:
            return True
    return False

selected_tickers = [t for t in tickers if _in_selected_tiers(market_caps.get(t))]
print(f"Tiers {CAP_TIERS_SELECTED}: {len(selected_tickers)} tickers")
data_filtered = data[selected_tickers]

scored = scorer_mr.compute_scores(
    data=data_filtered,
    sectors=sectors,
    options_df=options_df,
    market_caps=market_caps,
    top_n=TOP_N,
    max_per_sector=MAX_PER_SECTOR,
    min_score=MIN_SCORE,
    w_rsi=W_RSI, w_bb=W_BB, w_mr=W_MR, w_pcr=W_PCR,
    w_stochrsi=W_STOCHRSI, w_williams=W_WILLIAMS,
    rsi_period=RSI_PERIOD, bb_period=BB_PERIOD, bb_std=BB_STD,
    mr_period=MR_PERIOD, atr_period=ATR_PERIOD, volume_period=VOLUME_PERIOD,
    stoch_rsi_period=STOCH_RSI_PERIOD, stoch_smooth_k=STOCH_SMOOTH_K, stoch_smooth_d=STOCH_SMOOTH_D,
    williams_period=WILLIAMS_PERIOD, rsi_max=RSI_MAX, min_avg_volume=MIN_AVG_VOLUME,
)

scorer_mr.print_mr_report(scored)
ticker_list_df = scorer_mr.to_dataframe(scored)
ticker_list_df

## 3. BACKTEST

In [ ]:
#9 Regime Detector — Load over the ENTIRE universe range
LOAD_REGIME_DETECTOR = True   # True/False: load the regime detector

regime_detector = None
if LOAD_REGIME_DETECTOR:
    regime_detector = RegimeDetector()
    regime_start = data.index[0].strftime("%Y-%m-%d")
    regime_end   = data.index[-1].strftime("%Y-%m-%d")
    regime_detector.load(start=regime_start, end=regime_end, universe_data=data)
    print(f"✅ RegimeDetector loaded: {regime_start} -> {regime_end}")


In [ ]:
#10a Backtest comparison: with and without RegimeDetector
RUN_WITH_REGIME    = True
RUN_WITHOUT_REGIME = True
PRINT_COMPARISON   = True

# ── Ποιο sector-cap/sizing setup θες σταθερό για ΑΥΤΗ τη σύγκριση ──
REGIME_TEST_SECTORS = sectors               # sectors = ενεργό cap | None = χωρίς
REGIME_TEST_SIZING  = "fixed_fractional"    # "fixed_fractional" | "atr_risk_based"

scorer_weights_cfg = dict(
    w_rsi=W_RSI, w_bb=W_BB, w_mr=W_MR,
    w_stochrsi=W_STOCHRSI, w_williams=W_WILLIAMS,
    rsi_period=RSI_PERIOD, bb_period=BB_PERIOD, bb_std=BB_STD,
    mr_period=MR_PERIOD, atr_period=ATR_PERIOD, volume_period=VOLUME_PERIOD,
    stoch_rsi_period=STOCH_RSI_PERIOD, stoch_smooth_k=STOCH_SMOOTH_K, stoch_smooth_d=STOCH_SMOOTH_D,
    williams_period=WILLIAMS_PERIOD, rsi_max=RSI_MAX, min_avg_volume=MIN_AVG_VOLUME,
)
backtest_kwargs = dict(
    data=data,
    start_date=BACKTEST_START,
    end_date=BACKTEST_END,
    initial_capital=BACKTEST_CAPITAL,
    top_n=BACKTEST_TOP_N,
    signal_threshold=SIGNAL_THRESHOLD,
    exit_score_threshold=EXIT_SCORE_THRESHOLD,
    benchmark_ticker=BENCHMARK,
    max_hold_days=MAX_HOLD_DAYS,
    bear_regime_exit=BEAR_REGIME_EXIT,
    guard_days=GUARD_DAYS,
    hard_floor_atr=HARD_FLOOR_ATR,
    trail_trigger_atr=TRAIL_TRIGGER_ATR,
    atr_trail_mult=ATR_TRAIL_MULT,
    stop_atr_mult=STOP_ATR_MULT,
    target_atr_mult=TARGET_ATR_MULT,
    scorer_weights=scorer_weights_cfg,
    # sectors/sizing_method ΔΕΝ μπαίνουν εδώ — περνιούνται ρητά παρακάτω
    # ώστε τα #10b/#10c να μπορούν να κάνουν toggle χωρίς σύγκρουση kwargs.
)

results_with_regime = None
results_no_regime = None

sizing_extra = {}
if REGIME_TEST_SIZING == "atr_risk_based":
    sizing_extra = dict(
        risk_per_trade_pct=RISK_PER_TRADE_PCT,
        max_pct_per_position=MAX_PCT_PER_POSITION,
        liquidity_pct_adv=LIQUIDITY_PCT_ADV,
    )

if RUN_WITH_REGIME:
    print("█"*60)
    print("  RUN 1 — WITH RegimeDetector")
    print("█"*60)
    results_with_regime = backtester.run_backtest(
        regime_detector=regime_detector, **backtest_kwargs, membership=membership,
        sectors=REGIME_TEST_SECTORS, sizing_method=REGIME_TEST_SIZING, **sizing_extra,
    )
    backtester.print_backtest_report(results_with_regime)

if RUN_WITHOUT_REGIME:
    print("\n" + "█"*60)
    print("  RUN 2 — WITHOUT regime (fixed)")
    print("█"*60)
    results_no_regime = backtester.run_backtest(
        regime_detector=None, **backtest_kwargs, membership=membership,
        sectors=REGIME_TEST_SECTORS, sizing_method=REGIME_TEST_SIZING, **sizing_extra,
    )
    backtester.print_backtest_report(results_no_regime)

if PRINT_COMPARISON:
    if results_with_regime is None or results_no_regime is None:
        print("\n⚠️  Comparison skipped — needs both runs (RUN_WITH_REGIME=True and RUN_WITHOUT_REGIME=True).")
    else:
        print("\n" + "═"*60)
        print("  COMPARISON")
        print("═"*60)
        s1, s2 = results_with_regime["summary"], results_no_regime["summary"]
        print(f"{'Metric':<20} {'With regime':>15} {'Without regime':>15}")
        print(f"{'Total return':<20} {s1['total_return']:>+14.2f}% {s2['total_return']:>+14.2f}%")
        print(f"{'Annual return':<20} {s1['annual_return']:>+14.2f}% {s2['annual_return']:>+14.2f}%")
        print(f"{'Sharpe':<20} {s1['sharpe_ratio']:>15.2f} {s2['sharpe_ratio']:>15.2f}")
        print(f"{'Max drawdown':<20} {s1['max_drawdown']:>+14.2f}% {s2['max_drawdown']:>+14.2f}%")
        print(f"{'Sortino':<20} {s1['sortino_ratio']:>15.2f} {s2['sortino_ratio']:>15.2f}")
        print(f"{'Calmar':<20} {s1['calmar_ratio']:>15.2f} {s2['calmar_ratio']:>15.2f}")
        print(f"{'Expectancy':<20} {s1['expectancy_pct']:>+14.2f}% {s2['expectancy_pct']:>+14.2f}%")
        print(f"{'Win rate':<20} {s1['win_rate']:>14.1f}% {s2['win_rate']:>14.1f}%")
        print(f"{'N trades':<20} {s1['n_trades']:>15} {s2['n_trades']:>15}")

In [ ]:
#10b Sizing Method Comparison (fixed_fractional vs atr_risk_based)
# Sizing method είναι η μεταβλητή ΥΠΟ ΣΥΓΚΡΙΣΗ εδώ.
# Regime & sector cap είναι "σταθερά" ΜΟΝΟ με την έννοια ότι δεν αλλάζουν
# ανάμεσα στα δύο runs — ΕΣΥ διαλέγεις τις τιμές τους παρακάτω.

# ── Ποιο regime/sector-cap setup θες να κρατήσεις σταθερό για ΑΥΤΗ τη σύγκριση ──
SIZING_TEST_REGIME = None      # None = χωρίς regime | regime_detector = με regime
SIZING_TEST_SECTORS = sectors  # sectors = ενεργό sector cap | None = χωρίς cap

RUN_FIXED_SIZING        = True
RUN_ATR_SIZING          = True
PRINT_SIZING_COMPARISON = True

RISK_PER_TRADE_PCT   = 0.01
MAX_PCT_PER_POSITION = 0.20
LIQUIDITY_PCT_ADV     = 0.01

results_fixed = None
results_atr   = None

if RUN_FIXED_SIZING:
    print("█"*60)
    print(f"  RUN — Fixed-fractional sizing  (regime={'ON' if SIZING_TEST_REGIME else 'OFF'}, sector_cap={'ON' if SIZING_TEST_SECTORS else 'OFF'})")
    print("█"*60)
    results_fixed = backtester.run_backtest(
        **backtest_kwargs, membership=membership,
        regime_detector=SIZING_TEST_REGIME,
        sectors=SIZING_TEST_SECTORS,
        sizing_method="fixed_fractional",
    )
    backtester.print_backtest_report(results_fixed)

if RUN_ATR_SIZING:
    print("\n" + "█"*60)
    print(f"  RUN — ATR risk-based sizing  (regime={'ON' if SIZING_TEST_REGIME else 'OFF'}, sector_cap={'ON' if SIZING_TEST_SECTORS else 'OFF'})")
    print("█"*60)
    results_atr = backtester.run_backtest(
        **backtest_kwargs, membership=membership,
        regime_detector=SIZING_TEST_REGIME,
        sectors=SIZING_TEST_SECTORS,
        sizing_method="atr_risk_based",
        risk_per_trade_pct=RISK_PER_TRADE_PCT,
        max_pct_per_position=MAX_PCT_PER_POSITION,
        liquidity_pct_adv=LIQUIDITY_PCT_ADV,
    )
    backtester.print_backtest_report(results_atr)

if PRINT_SIZING_COMPARISON:
    if results_fixed is None or results_atr is None:
        print("\n⚠️  Comparison skipped — needs both runs (RUN_FIXED_SIZING=True and RUN_ATR_SIZING=True).")
    else:
        print("\n" + "═"*60)
        print("  SIZING COMPARISON")
        print("═"*60)
        s1, s2 = results_fixed["summary"], results_atr["summary"]
        print(f"{'Metric':<20} {'Fixed-fractional':>17} {'ATR risk-based':>17}")
        print(f"{'Total return':<20} {s1['total_return']:>+16.2f}% {s2['total_return']:>+16.2f}%")
        print(f"{'Annual return':<20} {s1['annual_return']:>+16.2f}% {s2['annual_return']:>+16.2f}%")
        print(f"{'Sharpe':<20} {s1['sharpe_ratio']:>17.2f} {s2['sharpe_ratio']:>17.2f}")
        print(f"{'Sortino':<20} {s1['sortino_ratio']:>17.2f} {s2['sortino_ratio']:>17.2f}")
        print(f"{'Calmar':<20} {s1['calmar_ratio']:>17.2f} {s2['calmar_ratio']:>17.2f}")
        print(f"{'Max drawdown':<20} {s1['max_drawdown']:>+16.2f}% {s2['max_drawdown']:>+16.2f}%")
        print(f"{'Expectancy':<20} {s1['expectancy_pct']:>+16.2f}% {s2['expectancy_pct']:>+16.2f}%")
        print(f"{'Win rate':<20} {s1['win_rate']:>16.1f}% {s2['win_rate']:>16.1f}%")
        print(f"{'N trades':<20} {s1['n_trades']:>17} {s2['n_trades']:>17}")

In [ ]:
#10c Sector Cap Comparison (with vs without diversification)
# Sector cap είναι η μεταβλητή ΥΠΟ ΣΥΓΚΡΙΣΗ εδώ (ON vs OFF).
# Regime & sizing method είναι "σταθερά" ΜΟΝΟ με την έννοια ότι ΔΕΝ αλλάζουν
# ανάμεσα στα δύο runs — αλλά ΕΣΥ διαλέγεις ποια τιμή θα έχουν παρακάτω.

# ── Ποιο regime/sizing setup θες να κρατήσεις σταθερό για ΑΥΤΗ τη σύγκριση ──
SECTOR_TEST_REGIME  = None              # None = χωρίς regime | regime_detector = με regime
SECTOR_TEST_SIZING  = "fixed_fractional"  # "fixed_fractional" | "atr_risk_based"

RUN_SECTOR_CAP_ON       = True
RUN_SECTOR_CAP_OFF      = True
PRINT_SECTOR_COMPARISON = True

sizing_extra = {}
if SECTOR_TEST_SIZING == "atr_risk_based":
    sizing_extra = dict(
        risk_per_trade_pct=RISK_PER_TRADE_PCT,
        max_pct_per_position=MAX_PCT_PER_POSITION,
        liquidity_pct_adv=LIQUIDITY_PCT_ADV,
    )

results_sector_on  = None
results_sector_off = None

if RUN_SECTOR_CAP_ON:
    print("█"*60)
    print(f"  RUN — Sector cap ON  (regime={'ON' if SECTOR_TEST_REGIME else 'OFF'}, sizing={SECTOR_TEST_SIZING})")
    print("█"*60)
    results_sector_on = backtester.run_backtest(
        **backtest_kwargs, membership=membership,
        regime_detector=SECTOR_TEST_REGIME,
        sizing_method=SECTOR_TEST_SIZING,
        sectors=sectors,
        **sizing_extra,
    )
    backtester.print_backtest_report(results_sector_on)

if RUN_SECTOR_CAP_OFF:
    print("\n" + "█"*60)
    print(f"  RUN — Sector cap OFF (regime={'ON' if SECTOR_TEST_REGIME else 'OFF'}, sizing={SECTOR_TEST_SIZING})")
    print("█"*60)
    results_sector_off = backtester.run_backtest(
        **backtest_kwargs, membership=membership,
        regime_detector=SECTOR_TEST_REGIME,
        sizing_method=SECTOR_TEST_SIZING,
        sectors=None,
        **sizing_extra,
    )
    backtester.print_backtest_report(results_sector_off)

if PRINT_SECTOR_COMPARISON:
    if results_sector_on is None or results_sector_off is None:
        print("\n⚠️  Comparison skipped — needs both runs.")
    else:
        print("\n" + "═"*60)
        print("  SECTOR CAP COMPARISON")
        print("═"*60)
        s1, s2 = results_sector_on["summary"], results_sector_off["summary"]
        print(f"{'Metric':<20} {'Cap ON':>15} {'Cap OFF':>15}")
        print(f"{'Total return':<20} {s1['total_return']:>+14.2f}% {s2['total_return']:>+14.2f}%")
        print(f"{'Annual return':<20} {s1['annual_return']:>+14.2f}% {s2['annual_return']:>+14.2f}%")
        print(f"{'Sharpe':<20} {s1['sharpe_ratio']:>15.2f} {s2['sharpe_ratio']:>15.2f}")
        print(f"{'Sortino':<20} {s1['sortino_ratio']:>15.2f} {s2['sortino_ratio']:>15.2f}")
        print(f"{'Calmar':<20} {s1['calmar_ratio']:>15.2f} {s2['calmar_ratio']:>15.2f}")
        print(f"{'Max drawdown':<20} {s1['max_drawdown']:>+14.2f}% {s2['max_drawdown']:>+14.2f}%")
        print(f"{'Expectancy':<20} {s1['expectancy_pct']:>+14.2f}% {s2['expectancy_pct']:>+14.2f}%")
        print(f"{'Win rate':<20} {s1['win_rate']:>14.1f}% {s2['win_rate']:>14.1f}%")
        print(f"{'N trades':<20} {s1['n_trades']:>15} {s2['n_trades']:>15}")

In [ ]:
#10d Correlation Filter Comparison (with vs without)
# Regime, sizing, sector cap κρατιούνται ΣΤΑΘΕΡΑ εδώ — απομονώνουμε ΜΟΝΟ
# τη μεταβλητή correlation filter, ίδια λογική με #10b/#10c.
CORR_TEST_REGIME  = None                 # None = χωρίς regime | regime_detector = με regime
CORR_TEST_SIZING  = "fixed_fractional"   # "fixed_fractional" | "atr_risk_based"
CORR_TEST_SECTORS = sectors              # sectors = ενεργό sector cap | None = χωρίς

MAX_PAIRWISE_CORRELATION  = 0.70
CORRELATION_LOOKBACK_DAYS = 60

RUN_CORR_FILTER_ON       = True
RUN_CORR_FILTER_OFF      = True
PRINT_CORR_COMPARISON    = True

sizing_extra = {}
if CORR_TEST_SIZING == "atr_risk_based":
    sizing_extra = dict(
        risk_per_trade_pct=RISK_PER_TRADE_PCT,
        max_pct_per_position=MAX_PCT_PER_POSITION,
        liquidity_pct_adv=LIQUIDITY_PCT_ADV,
    )

results_corr_on  = None
results_corr_off = None

if RUN_CORR_FILTER_ON:
    print("█"*60)
    print(f"  RUN — Correlation filter ON (max|corr|={MAX_PAIRWISE_CORRELATION})")
    print("█"*60)
    results_corr_on = backtester.run_backtest(
        **backtest_kwargs, membership=membership,
        regime_detector=CORR_TEST_REGIME,
        sizing_method=CORR_TEST_SIZING,
        sectors=CORR_TEST_SECTORS,
        max_pairwise_correlation=MAX_PAIRWISE_CORRELATION,
        correlation_lookback_days=CORRELATION_LOOKBACK_DAYS,
        **sizing_extra,
    )
    backtester.print_backtest_report(results_corr_on)

if RUN_CORR_FILTER_OFF:
    print("\n" + "█"*60)
    print("  RUN — Correlation filter OFF")
    print("█"*60)
    results_corr_off = backtester.run_backtest(
        **backtest_kwargs, membership=membership,
        regime_detector=CORR_TEST_REGIME,
        sizing_method=CORR_TEST_SIZING,
        sectors=CORR_TEST_SECTORS,
        max_pairwise_correlation=None,
        **sizing_extra,
    )
    backtester.print_backtest_report(results_corr_off)

if PRINT_CORR_COMPARISON:
    if results_corr_on is None or results_corr_off is None:
        print("\n⚠️  Comparison skipped — needs both runs.")
    else:
        print("\n" + "═"*60)
        print("  CORRELATION FILTER COMPARISON")
        print("═"*60)
        s1, s2 = results_corr_on["summary"], results_corr_off["summary"]
        print(f"{'Metric':<20} {'Filter ON':>15} {'Filter OFF':>15}")
        print(f"{'Total return':<20} {s1['total_return']:>+14.2f}% {s2['total_return']:>+14.2f}%")
        print(f"{'Annual return':<20} {s1['annual_return']:>+14.2f}% {s2['annual_return']:>+14.2f}%")
        print(f"{'Sharpe':<20} {s1['sharpe_ratio']:>15.2f} {s2['sharpe_ratio']:>15.2f}")
        print(f"{'Sortino':<20} {s1['sortino_ratio']:>15.2f} {s2['sortino_ratio']:>15.2f}")
        print(f"{'Calmar':<20} {s1['calmar_ratio']:>15.2f} {s2['calmar_ratio']:>15.2f}")
        print(f"{'Max drawdown':<20} {s1['max_drawdown']:>+14.2f}% {s2['max_drawdown']:>+14.2f}%")
        print(f"{'Expectancy':<20} {s1['expectancy_pct']:>+14.2f}% {s2['expectancy_pct']:>+14.2f}%")
        print(f"{'N trades':<20} {s1['n_trades']:>15} {s2['n_trades']:>15}")

## 4. BACKTEST DIAGNOSTICS

In [ ]:
#11 Stop Loss Forensics

RUN_FORENSICS       = False   # True/False: run the analysis
PRINT_REPORT        = False   # True/False: print the report
PLOT_DISTRIBUTIONS  = False   # True/False: plot_distributions() + plot_opportunity_cost()
SAVE_CSV            = False   # True/False: save stop_forensics.csv

RESULTS_SOURCE = "no_regime"   # "with_regime" or "no_regime" — which backtest run to analyse

if RUN_FORENSICS:
    from stop_loss_forensics import StopLossForensics

    if RESULTS_SOURCE == "with_regime":
        _results = globals().get("results_with_regime")
    elif RESULTS_SOURCE == "no_regime":
        _results = globals().get("results_no_regime")
    else:
        _results = globals().get("results")

    if _results is None:
        print(f"⚠️  No results found for RESULTS_SOURCE='{RESULTS_SOURCE}'. "
              f"Run the corresponding backtest first.")
    else:
        forensics = StopLossForensics(data, output_path=DATA_FOLDER)
        forensics.run(_results["trades"])

        if PRINT_REPORT:
            forensics.print_report()

        if PLOT_DISTRIBUTIONS:
            forensics.plot_distributions()
            forensics.plot_opportunity_cost()

        if SAVE_CSV:
            forensics.get_stops_df().to_csv(
                os.path.join(DATA_FOLDER, "stop_forensics.csv"), index=False
            )
            print("💾 Saved: stop_forensics.csv")


In [ ]:
#12 Trade-level & Survivorship Diagnostics
RESULTS_SOURCE = "no_regime"   # "with_regime" or "no_regime"

SHOW_1_FULL_TRADE_LIST        = False   # Full trade list, chronological
SHOW_2_CONCENTRATION          = False   # How many times the same ticker was entered
SHOW_3_DELISTING_CHECK        = False   # Which tickers "died" before the end of the dataset
SHOW_4_MEMBERSHIP_CROSSCHECK  = False   # Cross-check with membership table (index additions/removals)


if RESULTS_SOURCE == "with_regime":
    _results = globals().get("results_with_regime")
elif RESULTS_SOURCE == "no_regime":
    _results = globals().get("results_no_regime")
else:
    _results = globals().get("results")

if _results is None:
    print(f"⚠️  No results found for RESULTS_SOURCE='{RESULTS_SOURCE}'. "
          f"Run the corresponding backtest first.")
else:
    trades = _results['trades'].copy()
    trades = trades.sort_values('entry_date').reset_index(drop=True)

    # ── 1) Full trade list, chronological ──────────────────────────────
    if SHOW_1_FULL_TRADE_LIST:
        pd.set_option('display.max_rows', None)
        pd.set_option('display.width', 160)
        cols = ['ticker', 'entry_date', 'exit_date', 'entry_price', 'exit_price',
                'pnl_pct', 'hold_days', 'exit_reason', 'regime', 'signal_score']
        print(f"Total trades: {len(trades)}  |  Unique tickers: {trades['ticker'].nunique()}\n")
        print(trades[cols].to_string(index=False))

    # ── 2) How many times the same ticker was entered (concentration check) ───────
    if SHOW_2_CONCENTRATION:
        print(trades['ticker'].value_counts().to_string())

    # ── 3) Which traded tickers stopped having data before the end
    #    of the dataset — i.e. "died" (delisting/merger/bankruptcy)
    #    at some point, regardless of whether it coincided with an open position ────
    if SHOW_3_DELISTING_CHECK:
        last_date_overall = data.index.max()
        rows = []
        for t in trades['ticker'].unique():
            tclose = data[t]['Close'].dropna()
            last_dt = tclose.index.max() if not tclose.empty else None
            still_trading = (last_dt is not None) and (last_dt >= last_date_overall - pd.Timedelta(days=10))
            rows.append({
                'ticker': t,
                'last_data_date': last_dt,
                'still_trading_at_dataset_end': still_trading,
            })
        survivorship_check = pd.DataFrame(rows).sort_values('last_data_date')
        print(survivorship_check.to_string(index=False))
        n_dead = (~survivorship_check['still_trading_at_dataset_end']).sum()
        print(f"\n{n_dead} of {len(survivorship_check)} traded tickers stopped having data before the end of the dataset.")

    # ── 4) Cross-check with the membership table: which traded tickers left
    #    the index (S&P 500) at some point ─────────────────────────────
    if SHOW_4_MEMBERSHIP_CROSSCHECK:
        # Reuses the `membership` DataFrame already loaded in the
        # "Backtest configs" cell — no need to fetch it again.
        traded = trades['ticker'].unique()
        mem_traded = membership[membership['ticker'].isin(traded)]
        removed = mem_traded[mem_traded['date_removed'].notna()].sort_values('date_removed')
        print("Traded tickers that left the index membership table at some point:")
        print(removed[['ticker', 'date_added', 'date_removed']].to_string(index=False))
        print(f"\n{removed['ticker'].nunique()} of {len(traded)} traded tickers have ever left the index.")


In [ ]:
#13 Monte Carlo Simulations
from monte_carlo import run_monte_carlo, summarize_monte_carlo, print_monte_carlo_report, plot_monte_carlo_distribution

RUN_MONTE_CARLO   = False   # True/False: run the bootstrap resampling
PRINT_MC_REPORT   = False   # True/False: print the percentile/fragility report
PLOT_MC_DISTRIBUTION = False   # True/False: histogram of simulated total_return

N_SIMULATIONS = 2000
MC_METHOD     = "bootstrap"   # "bootstrap" or "shuffle
MC_SEED       = 42

if RUN_MONTE_CARLO:
    mc_df = run_monte_carlo(
        trades_df       = results_no_regime["trades"],
        initial_capital = BACKTEST_CAPITAL,
        top_n           = BACKTEST_TOP_N,
        n_simulations   = N_SIMULATIONS,
        method          = MC_METHOD,
        random_seed     = MC_SEED,
    )

    summary = summarize_monte_carlo(
        mc_df,
        known_total_return = results_no_regime["summary"]["total_return"],
        known_max_drawdown = results_no_regime["summary"]["max_drawdown"],
    )

    if PRINT_MC_REPORT:
        print_monte_carlo_report(summary)

    if PLOT_MC_DISTRIBUTION:
        plot_monte_carlo_distribution(mc_df, known_total_return=results_no_regime["summary"]["total_return"])

In [ ]:
#14 Rolling Windows Backtest
from out_of_sample import clamp_to_pre_oos, OOS_START

RUN_ROLLING_WINDOWS                = False    # True/False: run the rolling windows loop
USE_REGIME_DETECTOR                = False   # True: pass the global regime_detector | False: fixed mode (None)

PRINT_WINDOW_TABLE                 = False
PRINT_OUTPERFORMANCE_DISTRIBUTION  = False
PRINT_PERCENTILE_COMPARISON        = False
PRINT_TOP_BOTTOM                   = False

WINDOWS_START = "2011-01-01"
WINDOWS_END   = "2026-05-01"
WINDOWS_END = clamp_to_pre_oos(WINDOWS_END)
WINDOW_YEARS  = 2
STEP_MONTHS   = 6

KNOWN_OUTPERFORMANCE = 54.42


def run_rolling_windows(
    backtest_kwargs,
    regime_detector,
    windows_start = "2011-01-01",
    windows_end   = "2026-05-01",
    window_years  = 2,
    step_months   = 6,
):
    rows  = []
    start = pd.Timestamp(windows_start)
    final = pd.Timestamp(windows_end)

    while True:
        end = start + pd.DateOffset(years=window_years)
        if end > final:
            break

        label = f"{start.date()} -> {end.date()}"
        try:
            buf = io.StringIO()
            with contextlib.redirect_stdout(buf):
                res = backtester.run_backtest(
                    start_date      = start.strftime("%Y-%m-%d"),
                    end_date        = end.strftime("%Y-%m-%d"),
                    regime_detector = regime_detector,
                    **backtest_kwargs,
                )
            s = res["summary"]
            rows.append({
                "window_start":     start.date(),
                "window_end":       end.date(),
                **s,   # πλέον περιέχει sortino_ratio, calmar_ratio, expectancy_pct/abs αυτόματα
            })
            print(f"✅ {label}   outperf={s['outperformance']:+.1f}pp   trades={s['n_trades']}")
        except Exception as e:
            print(f"⚠️  {label} failed: {e}")

        start += pd.DateOffset(months=step_months)

    return pd.DataFrame(rows)


rolling_df = None

if RUN_ROLLING_WINDOWS:
    rolling_kwargs = dict(
        data=data,
        initial_capital=BACKTEST_CAPITAL,
        top_n=BACKTEST_TOP_N,
        signal_threshold=SIGNAL_THRESHOLD,
        exit_score_threshold=EXIT_SCORE_THRESHOLD,
        benchmark_ticker=BENCHMARK,
        max_hold_days=MAX_HOLD_DAYS,
        bear_regime_exit=BEAR_REGIME_EXIT,
        guard_days=GUARD_DAYS,
        hard_floor_atr=HARD_FLOOR_ATR,
        trail_trigger_atr=TRAIL_TRIGGER_ATR,
        atr_trail_mult=ATR_TRAIL_MULT,
        stop_atr_mult=STOP_ATR_MULT,
        target_atr_mult=TARGET_ATR_MULT,
        scorer_weights=scorer_weights_cfg,
        membership=membership,
    )

    # regime_detector is already loaded, in the loading cell above, over the
    # ENTIRE range of data — no need to reload / regime_detector_full here.
    _regime_arg = regime_detector if USE_REGIME_DETECTOR else None

    rolling_df = run_rolling_windows(
        rolling_kwargs,
        _regime_arg,
        windows_start = WINDOWS_START,
        windows_end   = WINDOWS_END,
        window_years  = WINDOW_YEARS,
        step_months   = STEP_MONTHS,
    )

if rolling_df is None:
    if any([PRINT_WINDOW_TABLE, PRINT_OUTPERFORMANCE_DISTRIBUTION,
            PRINT_PERCENTILE_COMPARISON, PRINT_TOP_BOTTOM]):
        print("⚠️  No rolling_df — set RUN_ROLLING_WINDOWS=True to run the loop.")
else:
    if PRINT_WINDOW_TABLE:
        pd.set_option('display.max_rows', None)
        print(rolling_df[[
            "window_start","window_end","total_return","benchmark_return",
            "outperformance","n_trades","sharpe_ratio","sortino_ratio",
            "calmar_ratio","max_drawdown","expectancy_pct",
        ]].to_string(index=False))

    if PRINT_OUTPERFORMANCE_DISTRIBUTION:
        print(f"\n{'='*50}")
        print("OUTPERFORMANCE DISTRIBUTION (pp)")
        print(f"{'='*50}")
        print(rolling_df["outperformance"].describe())

    if PRINT_PERCENTILE_COMPARISON:
        pct_rank = (rolling_df["outperformance"] < KNOWN_OUTPERFORMANCE).mean() * 100
        print(f"\nThe known outperformance (+{KNOWN_OUTPERFORMANCE}pp) is at the {pct_rank:.0f}th percentile of {len(rolling_df)} windows.")

    if PRINT_TOP_BOTTOM:
        cols = ["window_start","window_end","outperformance","n_trades","sortino_ratio","max_drawdown"]
        print(f"\nTop 5 best windows:")
        print(rolling_df.nlargest(5, "outperformance")[cols].to_string(index=False))
        print(f"\nTop 5 worst windows:")
        print(rolling_df.nsmallest(5, "outperformance")[cols].to_string(index=False))

In [ ]:
#15 Parameter Sweep (Stop-Loss & Filter Grid Search)
from parameter_sweep import run_parameter_sweep, pivot_sweep_results, plot_sweep_heatmap, analyze_local_stability
from out_of_sample import clamp_to_pre_oos

RUN_PARAMETER_SWEEP  = False   # True/False: run the grid search
PLOT_SWEEP_HEATMAP   = False   # True/False: show the heatmap
ANALYZE_STABILITY    = False   # True/False: run the local-stability check on the current values

SWEEP_METRIC = "sharpe_ratio"
SWEEP_GRID = {
    "atr_trail_mult":    [1.5, 2.0, 2.5, 3.0],
    "trail_trigger_atr": [0.5, 1.0, 1.5],
}

if RUN_PARAMETER_SWEEP:
    sweep_kwargs = dict(backtest_kwargs)
    sweep_kwargs["end_date"] = clamp_to_pre_oos(sweep_kwargs["end_date"])

    sweep_df = run_parameter_sweep(
        backtester_module = backtester,
        base_kwargs         = sweep_kwargs,
        param_grid           = SWEEP_GRID,
        regime_detector       = regime_detector,
        membership            = membership,
    )

    pivot = pivot_sweep_results(
        sweep_df, param_x="atr_trail_mult", param_y="trail_trigger_atr", metric=SWEEP_METRIC,
    )

    if PLOT_SWEEP_HEATMAP:
        plot_sweep_heatmap(
            pivot, metric=SWEEP_METRIC,
            current_x=ATR_TRAIL_MULT, current_y=TRAIL_TRIGGER_ATR,
        )

    if ANALYZE_STABILITY:
        analyze_local_stability(
            pivot, current_x=ATR_TRAIL_MULT, current_y=TRAIL_TRIGGER_ATR, metric=SWEEP_METRIC,
        )

In [ ]:
#16 Transaction Cost Stress Test
from cost_stress_test import run_cost_stress_test, print_stress_test_report

RUN_STRESS_TEST = False

if RUN_STRESS_TEST:
    stress_df, stress_results = run_cost_stress_test(
        backtester_module = backtester,
        base_kwargs        = backtest_kwargs,
        regime_detector    = regime_detector,
        membership         = membership,
        multipliers        = (1, 2, 3),
    )
    print_stress_test_report(stress_df)

In [ ]:
#17 Equal-Weight Benchmark (Same Universe as Scanner)
from equal_weight_benchmark import run_equal_weight_comparison, print_equal_weight_report

RUN_EQUAL_WEIGHT_BENCHMARK = False   # True/False: build the equal-weight comparison
PRINT_EQUAL_WEIGHT_REPORT  = False   # True/False: print the 3-way attribution report

if RUN_EQUAL_WEIGHT_BENCHMARK:
    ew_result = run_equal_weight_comparison(
        data              = data,
        trades_df         = results_no_regime["trades"],
        strategy_summary  = results_no_regime["summary"],
        start_date        = BACKTEST_START,
        end_date          = BACKTEST_END,
        initial_capital   = BACKTEST_CAPITAL,
    )

    if PRINT_EQUAL_WEIGHT_REPORT:
        print_equal_weight_report(
            strategy_summary      = results_no_regime["summary"],
            equal_weight_summary  = ew_result["summary"],
            benchmark_return      = results_no_regime["summary"]["benchmark_return"],   # SPY
            benchmark_ticker      = BENCHMARK,
        )

In [ ]:
#18 Option Move Backtest (based on signals from the MR backtest)
RESULTS_SOURCE = "no_regime"   # "with_regime" or "no_regime" — which run supplies the entry signals

RUN_OPTION_BACKTEST        = False   # True/False: run run_option_move_backtest
PRINT_SUMMARY_REPORT       = False   # True/False: print_option_backtest_report
SHOW_TRAILING_STOP_STATS   = False   # True/False: describe() on trailing_stop trades
SHOW_TOP5_TRAILING_STOP    = False   # True/False: top 5 best trailing_stop trades

results = None
summary = None

if RESULTS_SOURCE == "with_regime":
    trades_source_results = globals().get("results_with_regime")
elif RESULTS_SOURCE == "no_regime":
    trades_source_results = globals().get("results_no_regime")
else:
    trades_source_results = None

if trades_source_results is None:
    print(f"⚠️  No results found for RESULTS_SOURCE='{RESULTS_SOURCE}'. "
          f"Run the corresponding backtest first.")
else:
    trades_source = trades_source_results["trades"]
    signals_df = trades_source[["ticker", "entry_date", "signal_score"]].rename(
        columns={"entry_date": "signal_date"}
    ).reset_index(drop=True)

    if RUN_OPTION_BACKTEST:
        results = run_option_move_backtest(data, signals_df)
        summary = summarize_option_backtest(results)

        if PRINT_SUMMARY_REPORT:
            print_option_backtest_report(summary)

if results is None:
    if any([SHOW_TRAILING_STOP_STATS, SHOW_TOP5_TRAILING_STOP]):
        print("⚠️  No results — set RUN_OPTION_BACKTEST=True (and the correct RESULTS_SOURCE).")
else:
    if SHOW_TRAILING_STOP_STATS:
        print(results[results["exit_reason"] == "trailing_stop"]["option_pnl_pct"].describe())

    if SHOW_TOP5_TRAILING_STOP:
        print(results[results["exit_reason"] == "trailing_stop"].nlargest(5, "option_pnl_pct")[
            ["ticker", "signal_date", "option_pnl_pct", "days_held"]
        ])


## 5. LIVE SIGNAL TOOLS

In [ ]:
#19 Point-in-time Signal Inspector
def inspect_signal(ticker, date, data, sectors, scorer_weights=None):
    """
    Shows the full scorer breakdown for a ticker, as the scanner would have
    seen it AT THE TIME of `date` — point-in-time, no look-ahead (same logic
    as the backtester's _compute_mr_score_at_date).
    """
    date = pd.Timestamp(date)
    ticker_data = data[[ticker]].loc[:date]   # only up to that day
    w = scorer_weights or {}
    scored = scorer_mr.compute_scores(
        data=ticker_data,
        sectors={ticker: sectors.get(ticker, "Unknown")},
        options_df=None,            # no PCR history available -> no-PCR weights
        top_n=1, max_per_sector=1,
        min_score=-999,              # so nothing gets filtered out
        w_rsi=w.get("w_rsi"), w_bb=w.get("w_bb"), w_mr=w.get("w_mr"),
        w_stochrsi=w.get("w_stochrsi"), w_williams=w.get("w_williams"),
        rsi_period=w.get("rsi_period"), bb_period=w.get("bb_period"), bb_std=w.get("bb_std"),
        mr_period=w.get("mr_period"), atr_period=w.get("atr_period"),
        volume_period=w.get("volume_period"),
        stoch_rsi_period=w.get("stoch_rsi_period"),
        stoch_smooth_k=w.get("stoch_smooth_k"), stoch_smooth_d=w.get("stoch_smooth_d"),
        williams_period=w.get("williams_period"), rsi_max=w.get("rsi_max"),
        min_avg_volume=w.get("min_avg_volume"),
    )
    if not scored:
        print(f"⚠️ {ticker} did not pass hard filters on {date.date()} (RSI>max or low volume)")
        return None
    scorer_mr.print_mr_report(scored)
    return scored[0]


# ── Call settings ──────────────────────────────────────────────────────────
RUN_INSPECT_SIGNAL = False   # True/False: call inspect_signal below

INSPECT_TICKER = "MA"
INSPECT_DATE   = "2026-07-24"

detail = None
if RUN_INSPECT_SIGNAL:
    detail = inspect_signal(INSPECT_TICKER, INSPECT_DATE, data, sectors, scorer_weights_cfg)


In [ ]:
#20 Batch Signal Inspection (scorer characteristics of option trades)
import io
import pandas as pd
from contextlib import redirect_stdout

def inspect_signal_quiet(ticker, date, data, sectors, scorer_weights=None):
    """Same as inspect_signal, but without prints — for batch use."""
    date = pd.Timestamp(date)
    ticker_data = data[[ticker]].loc[:date]
    w = scorer_weights or {}
    with redirect_stdout(io.StringIO()):  # swallows the FILTER SUMMARY prints
        scored = scorer_mr.compute_scores(
            data=ticker_data,
            sectors={ticker: sectors.get(ticker, "Unknown")},
            options_df=None,
            top_n=1, max_per_sector=1, min_score=-999,
            w_rsi=w.get("w_rsi"), w_bb=w.get("w_bb"), w_mr=w.get("w_mr"),
            w_stochrsi=w.get("w_stochrsi"), w_williams=w.get("w_williams"),
            rsi_period=w.get("rsi_period"), bb_period=w.get("bb_period"), bb_std=w.get("bb_std"),
            mr_period=w.get("mr_period"), atr_period=w.get("atr_period"),
            volume_period=w.get("volume_period"),
            stoch_rsi_period=w.get("stoch_rsi_period"),
            stoch_smooth_k=w.get("stoch_smooth_k"), stoch_smooth_d=w.get("stoch_smooth_d"),
            williams_period=w.get("williams_period"), rsi_max=w.get("rsi_max"),
            min_avg_volume=w.get("min_avg_volume"),
        )
    return scored[0] if scored else None


def batch_inspect(trades_subset, data, sectors, scorer_weights=None):
    """
    Runs inspect_signal_quiet over every trade in trades_subset and joins
    the scorer characteristics with option_pnl_pct.
    """
    rows = []
    for _, trade in trades_subset.iterrows():
        detail = inspect_signal_quiet(
            trade["ticker"], trade["signal_date"], data, sectors, scorer_weights
        )
        if detail is None:
            continue
        rows.append({
            "ticker":          trade["ticker"],
            "signal_date":     trade["signal_date"],
            "option_pnl_pct":  trade["option_pnl_pct"],
            "exit_reason":     trade["exit_reason"],
            "days_held":       trade["days_held"],
            "composite_score": detail["composite_score"],
            "rsi":             detail["rsi"],
            "pct_b":           detail["pct_b"],
            "z_score":         detail["z_score"],
            "stoch_k":         detail["stoch_k"],
            "williams_r":      detail["williams_r"],
            "atr_percentile":  detail["atr_percentile"],
            "atr_regime":      detail["atr_pct_label"],
            "vol_ratio":       detail["vol_ratio"],
        })
    return pd.DataFrame(rows)


# ── Settings ─────────────────────────────────────────────────────────────────
RUN_BATCH_INSPECT      = False   # True/False: run batch_inspect over `results`
SHOW_TOP10_BY_PNL      = False   # True/False: top 10 trades by option_pnl_pct
SHOW_CORRELATION       = False   # True/False: correlation option_pnl_pct vs scorer features
SHOW_GROUPBY_EXIT_ATR  = False   # True/False: describe() atr_percentile per exit_reason

detail_df = None

if RUN_BATCH_INSPECT:
    if globals().get("results") is None:
        print("⚠️  `results` not found — run the option backtest cell first.")
    else:
        detail_df = batch_inspect(results, data, sectors, scorer_weights_cfg)

if detail_df is None:
    if any([SHOW_TOP10_BY_PNL, SHOW_CORRELATION, SHOW_GROUPBY_EXIT_ATR]):
        print("⚠️  No detail_df — set RUN_BATCH_INSPECT=True.")
else:
    if SHOW_TOP10_BY_PNL:
        print(detail_df.sort_values("option_pnl_pct", ascending=False).head(10))

    if SHOW_CORRELATION:
        print(detail_df[["option_pnl_pct", "atr_percentile", "rsi", "z_score", "williams_r"]].corr()["option_pnl_pct"])

    if SHOW_GROUPBY_EXIT_ATR:
        print(detail_df.groupby("exit_reason")["atr_percentile"].describe())


In [ ]:
#21a Feature vs Exit-Reason Breakdown (after batch_inspect) - Options
"""This cell is a diagnostic on which entry signal predicts a good/bad exit
— i.e. whether the depth of the mean-reversion signal (z_score) or other
scorer characteristics have predictive value for how the trade turns out"""

SHOW_FEATURE_STATS_BY_EXIT   = False   # True/False: describe(mean/std/50%) per feature x exit_reason
SHOW_ZBUCKET_EXIT_PCT        = False   # True/False: % distribution of exit_reason per z_score bucket
SHOW_ZBUCKET_EXIT_COUNTS     = False   # True/False: raw counts of exit_reason per z_score bucket

FEATURE_COLS = ["rsi", "z_score", "williams_r", "vol_ratio", "composite_score"]
Z_BUCKET_BINS = [-5, -2.5, -2.0, -1.5, -1.0, 0]

if globals().get("detail_df") is None:
    if any([SHOW_FEATURE_STATS_BY_EXIT, SHOW_ZBUCKET_EXIT_PCT, SHOW_ZBUCKET_EXIT_COUNTS]):
        print("⚠️  No detail_df — run the batch_inspect cell first.")
else:
    if SHOW_FEATURE_STATS_BY_EXIT:
        for col in FEATURE_COLS:
            print(f"\n── {col} ──")
            print(detail_df.groupby("exit_reason")[col].describe()[["mean", "std", "50%"]])

    if SHOW_ZBUCKET_EXIT_PCT or SHOW_ZBUCKET_EXIT_COUNTS:
        detail_df["z_bucket"] = pd.cut(detail_df["z_score"], bins=Z_BUCKET_BINS)

    if SHOW_ZBUCKET_EXIT_PCT:
        print("\n── % distribution of exit_reason per z_bucket ──")
        print(detail_df.groupby("z_bucket")["exit_reason"].value_counts(normalize=True).unstack().round(2))

    if SHOW_ZBUCKET_EXIT_COUNTS:
        print("\n── Counts of exit_reason per z_bucket ──")
        print(detail_df.groupby("z_bucket")["exit_reason"].value_counts().unstack())


In [ ]:
#21b Feature vs Exit-Reason / PnL Breakdown — STOCK ONLY (no options)

RESULTS_SOURCE_STOCK = "no_regime"   # "with_regime" or "no_regime"

RUN_STOCK_BATCH_INSPECT    = False
SHOW_STOCK_FEATURE_STATS   = False   # describe(mean/std/50%) per feature x exit_reason
SHOW_STOCK_ZBUCKET_PCT     = False   # % distribution of exit_reason per z_bucket
SHOW_STOCK_ZBUCKET_COUNTS  = False   # raw counts of exit_reason per z_bucket
SHOW_STOCK_ZBUCKET_PNL     = False   # mean/median pnl_pct per z_bucket (πιο άμεσο test)

FEATURE_COLS_STOCK  = ["rsi", "z_score", "williams_r", "vol_ratio", "composite_score"]
Z_BUCKET_BINS_STOCK = [-5, -2.5, -2.0, -1.5, -1.0, 0]


def batch_inspect_stock(trades_subset, data, sectors, scorer_weights=None):
    """Ίδια λογική με batch_inspect() (Κελί #20), αλλά πάνω στα ΠΡΑΓΜΑΤΙΚΑ
    stock trades — δεν χρειάζεται καθόλου το option_move_tracker."""
    rows = []
    for _, trade in trades_subset.iterrows():
        detail = inspect_signal_quiet(
            trade["ticker"], trade["entry_date"], data, sectors, scorer_weights
        )
        if detail is None:
            continue
        rows.append({
            "ticker":          trade["ticker"],
            "entry_date":      trade["entry_date"],
            "pnl_pct":         trade["pnl_pct"],
            "exit_reason":     trade["exit_reason"],
            "hold_days":       trade["hold_days"],
            "composite_score": detail["composite_score"],
            "rsi":             detail["rsi"],
            "pct_b":           detail["pct_b"],
            "z_score":         detail["z_score"],
            "stoch_k":         detail["stoch_k"],
            "williams_r":      detail["williams_r"],
            "atr_percentile":  detail["atr_percentile"],
            "atr_regime":      detail["atr_pct_label"],
            "vol_ratio":       detail["vol_ratio"],
        })
    return pd.DataFrame(rows)


if RESULTS_SOURCE_STOCK == "with_regime":
    _stock_results = globals().get("results_with_regime")
elif RESULTS_SOURCE_STOCK == "no_regime":
    _stock_results = globals().get("results_no_regime")
else:
    _stock_results = None

detail_df_stock = None
if RUN_STOCK_BATCH_INSPECT:
    if _stock_results is None:
        print(f"⚠️  No results found for RESULTS_SOURCE_STOCK='{RESULTS_SOURCE_STOCK}'. "
              f"Run the corresponding backtest first.")
    else:
        detail_df_stock = batch_inspect_stock(_stock_results["trades"], data, sectors, scorer_weights_cfg)

if detail_df_stock is None:
    if any([SHOW_STOCK_FEATURE_STATS, SHOW_STOCK_ZBUCKET_PCT, SHOW_STOCK_ZBUCKET_COUNTS, SHOW_STOCK_ZBUCKET_PNL]):
        print("⚠️  No detail_df_stock — set RUN_STOCK_BATCH_INSPECT=True.")
else:
    if SHOW_STOCK_FEATURE_STATS:
        for col in FEATURE_COLS_STOCK:
            print(f"\n── {col} ──")
            print(detail_df_stock.groupby("exit_reason")[col].describe()[["mean", "std", "50%"]])

    if SHOW_STOCK_ZBUCKET_PCT or SHOW_STOCK_ZBUCKET_COUNTS or SHOW_STOCK_ZBUCKET_PNL:
        detail_df_stock["z_bucket"] = pd.cut(detail_df_stock["z_score"], bins=Z_BUCKET_BINS_STOCK)

    if SHOW_STOCK_ZBUCKET_PCT:
        print("\n── % distribution of exit_reason per z_bucket (STOCK, no options) ──")
        print(detail_df_stock.groupby("z_bucket")["exit_reason"].value_counts(normalize=True).unstack().round(2))

    if SHOW_STOCK_ZBUCKET_COUNTS:
        print("\n── Counts of exit_reason per z_bucket (STOCK, no options) ──")
        print(detail_df_stock.groupby("z_bucket")["exit_reason"].value_counts().unstack())

    if SHOW_STOCK_ZBUCKET_PNL:
        print("\n── Mean / median pnl_pct per z_bucket (STOCK, no options) ──")
        print(detail_df_stock.groupby("z_bucket")["pnl_pct"].agg(["count", "mean", "median"]))

In [ ]:
#22 Custom Watchlist Scoring
RUN_CUSTOM_WATCHLIST = False   # True/False: σκόραρε custom λίστα tickers εκτός universe

if RUN_CUSTOM_WATCHLIST:
    custom_tickers = ["BABA", "NBIS"]   # πρόσθεσε ό,τι θες εδώ
    custom_sectors, custom_caps = get_sectors_and_caps(
        custom_tickers, folder_path="data", universe_name="custom_watchlist"
    )
    custom_data = download_sp500_data(
        custom_tickers, folder_path="data", universe_name="custom_watchlist"
    )
    custom_scored = scorer_mr.compute_scores(
        data           = custom_data,
        sectors        = custom_sectors,
        market_caps    = custom_caps,
        min_score      = -999,
        max_per_sector = 99,
        top_n          = 99,
    )
    scorer_mr.print_mr_report(custom_scored)
    #detail = inspect_signal("BABA", "2026-07-15", custom_data, custom_sectors, scorer_weights_cfg)


In [ ]:
#23 Daily Score History: single ticker, no simulated trades
import matplotlib.pyplot as plt
def compute_daily_score_history(ticker, scorer_weights=None, folder_path="data", period="5y"):
    
    from scorer_mr import _rsi as _compute_rsi
    from scorer_mr import _bollinger as _compute_bollinger
    from scorer_mr import _atr as _compute_atr
    from signals import _compute_stoch_rsi, _compute_williams_r
    import numpy as np

    ticker_data = download_sp500_data(
        [ticker], period=period, folder_path=folder_path, universe_name=f"{ticker.lower()}_solo"
    )
    df = ticker_data[ticker].dropna()

    w = scorer_weights or {}
    rsi_period       = w.get("rsi_period",       14)
    bb_period        = w.get("bb_period",         20)
    bb_std           = w.get("bb_std",            2.0)
    mr_period        = w.get("mr_period",         252)
    atr_period       = w.get("atr_period",        14)
    volume_period    = w.get("volume_period",     20)
    stoch_rsi_period = w.get("stoch_rsi_period",  14)
    stoch_smooth_k   = w.get("stoch_smooth_k",    3)
    stoch_smooth_d   = w.get("stoch_smooth_d",    3)
    williams_period  = w.get("williams_period",   14)
    rsi_max          = w.get("rsi_max",           50)
    min_avg_volume   = w.get("min_avg_volume",    500_000)

    close, high, low, volume = df["Close"], df["High"], df["Low"], df["Volume"]

    # ── Δείκτες — υπολογισμός ΜΙΑ φορά, vectorized, causal ────────────────────
    rsi_series             = _compute_rsi(close, rsi_period)
    _, _, pct_b_series, _  = _compute_bollinger(close, bb_period, bb_std)
    atr_series              = _compute_atr(high, low, close, atr_period)
    stoch_k_series, _       = _compute_stoch_rsi(
        close, rsi_period=rsi_period, stoch_period=stoch_rsi_period,
        smooth_k=stoch_smooth_k, smooth_d=stoch_smooth_d,
    )
    wr_series                = _compute_williams_r(high, low, close, williams_period)

    # z-score: expanding window στον πρώτο χρόνο, ίδιο με original scalar code
    rolling_mean = close.rolling(mr_period, min_periods=1).mean()
    rolling_std  = close.rolling(mr_period, min_periods=1).std()
    z_series     = (close - rolling_mean) / rolling_std.replace(0, np.nan)

    avg_vol_series = volume.rolling(volume_period).mean()

    # ── Vectorized scoring functions — ΙΔΙΑ breakpoints με scorer_mr.py ───────
    def _vscore_rsi(v):
        s = np.select(
            [v <= 20, v <= 25, v <= 30, v <= 35, v <= 40, v <= 45, v <= 50],
            [10.0, 9.0, 8.0, 7.0, 5.5, 3.5, 2.0], default=0.0,
        )
        return s

    def _vscore_bb(v):
        s = np.select(
            [v < -0.1, v < 0.0, v < 0.10, v < 0.20, v < 0.35, v < 0.65, v < 0.80],
            [10.0, 9.0, 8.0, 6.5, 5.0, 3.0, 2.0], default=1.0,
        )
        return np.where(pd.isna(v), 5.0, s)

    def _vscore_mr(v):
        s = np.select(
            [v < -2.5, v < -2.0, v < -1.5, v < -1.0, v < -0.5, v < 0.0],
            [10.0, 9.0, 7.5, 6.0, 4.0, 2.5], default=1.0,
        )
        return np.where(pd.isna(v), 5.0, s)

    def _vscore_stochrsi(v):
        s = np.select(
            [v <= 0.05, v <= 0.10, v <= 0.20, v <= 0.30, v <= 0.50, v <= 0.70, v <= 0.80],
            [10.0, 9.0, 8.0, 6.5, 4.0, 2.5, 1.5], default=0.5,
        )
        return np.where(pd.isna(v), 5.0, s)

    def _vscore_williams(v):
        s = np.select(
            [v <= -95, v <= -90, v <= -80, v <= -70, v <= -50, v <= -30, v <= -20],
            [10.0, 9.0, 8.0, 6.5, 4.0, 2.5, 1.5], default=0.5,
        )
        return np.where(pd.isna(v), 5.0, s)

    s_rsi      = _vscore_rsi(rsi_series.values)
    s_bb       = _vscore_bb(pct_b_series.values)
    s_mr       = _vscore_mr(z_series.values)
    s_stochrsi = _vscore_stochrsi(stoch_k_series.values)
    s_williams = _vscore_williams(wr_series.values)

    # ── Weights: no-PCR mode, renormalized στα 5 — ΙΔΙΟ με backtester ────────
    _w_rsi      = w.get("w_rsi",      0.30)
    _w_bb       = w.get("w_bb",       0.20)
    _w_mr       = w.get("w_mr",       0.15)
    _w_stochrsi = w.get("w_stochrsi", 0.10)
    _w_williams = w.get("w_williams", 0.10)
    total = _w_rsi + _w_bb + _w_mr + _w_stochrsi + _w_williams
    _w_rsi, _w_bb, _w_mr, _w_stochrsi, _w_williams = (
        _w_rsi / total, _w_bb / total, _w_mr / total,
        _w_stochrsi / total, _w_williams / total,
    )

    composite = (
        s_rsi * _w_rsi + s_bb * _w_bb + s_mr * _w_mr +
        s_stochrsi * _w_stochrsi + s_williams * _w_williams
    )

    # ── Hard filters (vectorized boolean masks, ίδια σειρά ελέγχου) ──────────
    warmup_needed = max(mr_period, bb_period, rsi_period, volume_period) + 5
    idx_pos = np.arange(len(df))

    valid_mask = (
        (idx_pos >= warmup_needed) &
        (close.values > 0) & ~pd.isna(close.values) &
        (avg_vol_series.values >= min_avg_volume) & ~pd.isna(avg_vol_series.values) &
        (rsi_series.values <= rsi_max) &
        (atr_series.values > 0) & ~pd.isna(atr_series.values)
    )

    reason = np.full(len(df), "ok", dtype=object)
    reason[idx_pos < warmup_needed] = "warmup"
    past_warmup = idx_pos >= warmup_needed
    reason[past_warmup & (rsi_series.values > rsi_max)] = "rsi_too_high"
    reason[past_warmup & (rsi_series.values <= rsi_max) & ~valid_mask] = "other"

    score_final = np.where(valid_mask, composite, np.nan)

    history = pd.DataFrame({
        "price":   close.values,
        "rsi":     rsi_series.values,
        "pct_b":   pct_b_series.values,
        "z_score": z_series.values,
        "score":   score_final,
        "reason":  reason,
    }, index=df.index)

    return history


def plot_daily_score_history(history, ticker, threshold=6.5):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True,
                                     gridspec_kw={"height_ratios": [2, 1]})

    ax1.plot(history.index, history["price"], color="black", linewidth=1)
    signal_days = history[history["score"] >= threshold]
    ax1.scatter(signal_days.index, signal_days["price"], color="green", marker="^",
                s=45, zorder=5, label=f"Score >= {threshold}")
    ax1.set_title(f"{ticker} — Price vs. Daily MR Score (5y)")
    ax1.set_ylabel("Price ($)")
    ax1.legend(loc="upper left", fontsize=8)
    ax1.grid(alpha=0.3)

    ax2.plot(history.index, history["score"], color="darkorange", linewidth=1)
    ax2.axhline(threshold, color="green", linestyle="--", alpha=0.6, label=f"signal_threshold={threshold}")
    ax2.set_ylabel("MR Score")
    ax2.set_ylim(0, 10)
    ax2.legend(loc="upper left", fontsize=8)
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    valid   = history["score"].notna().sum()
    above   = (history["score"] >= threshold).sum()
    warmup  = (history["reason"] == "warmup").sum()
    rsi_hi  = (history["reason"] == "rsi_too_high").sum()
    other   = (history["reason"] == "other").sum()
    print(f"Days with a valid score: {valid} / {len(history)} ({valid/len(history)*100:.1f}%)")
    print(f"  Excluded — warmup (insufficient history): {warmup}")
    print(f"  Excluded — RSI > max:                     {rsi_hi}")
    print(f"  Excluded — other (volume/ATR/etc):         {other}")
    print(f"Days scoring >= {threshold}: {above} ({above/len(history)*100:.1f}% of valid days)")


def plot_daily_score_history_interactive(history, ticker, threshold=6.5):
    
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    signal_days = history[history["score"] >= threshold]

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06,
        row_heights=[0.65, 0.35],
        subplot_titles=(f"{ticker} — Price", "MR Composite Score"),
    )

    # Row 1: Price line — customdata: [score, rsi, pct_b, z_score]
    fig.add_trace(
        go.Scatter(
            x=history.index, y=history["price"], name="Price",
            line=dict(color="black", width=1.3),
            customdata=history[["score", "rsi", "pct_b", "z_score"]].values,
            hovertemplate=(
                "Date: %{x|%d/%m/%Y}<br>"
                "Price: $%{y:.2f}<br>"
                "Score: %{customdata[0]:.1f}<br>"
                "RSI: %{customdata[1]:.0f}<br>"
                "BB%%: %{customdata[2]:.2f}<br>"
                "Z-Score: %{customdata[3]:.2f}<extra></extra>"
            ),
        ),
        row=1, col=1,
    )
    # Green markers on the price line for days that crossed the threshold —
    # so you can see directly on the price chart where a signal would fire.
    fig.add_trace(
        go.Scatter(
            x=signal_days.index, y=signal_days["price"], name=f"Score >= {threshold}",
            mode="markers", marker=dict(color="green", size=7, symbol="triangle-up"),
            hovertemplate="Signal day<br>Date: %{x|%d/%m/%Y}<br>Price: $%{y:.2f}<extra></extra>",
        ),
        row=1, col=1,
    )

    # Row 2: Score line + threshold reference line
    fig.add_trace(
        go.Scatter(
            x=history.index, y=history["score"], name="MR Score",
            line=dict(color="darkorange", width=1.3),
            hovertemplate="Score: %{y:.1f}<extra></extra>",
        ),
        row=2, col=1,
    )
    fig.add_hline(
        y=threshold, line_dash="dash", line_color="green", opacity=0.6,
        annotation_text=f"threshold={threshold}", annotation_position="top left",
        row=2, col=1,
    )

    fig.update_layout(
        height=650, hovermode="x unified",
        title=f"{ticker} — Price vs. Daily MR Score (interactive)",
        legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="left", x=0),
    )
    fig.update_yaxes(title_text="Price ($)", row=1, col=1)
    fig.update_yaxes(title_text="Score", range=[0, 10], row=2, col=1)
    fig.update_xaxes(rangeslider=dict(visible=True, thickness=0.05), row=2, col=1)

    fig.show()

    valid   = history["score"].notna().sum()
    above   = (history["score"] >= threshold).sum()
    warmup  = (history["reason"] == "warmup").sum()
    rsi_hi  = (history["reason"] == "rsi_too_high").sum()
    other   = (history["reason"] == "other").sum()
    print(f"Days with a valid score: {valid} / {len(history)} ({valid/len(history)*100:.1f}%)")
    print(f"  Excluded — warmup (insufficient history): {warmup}")
    print(f"  Excluded — RSI > max:                     {rsi_hi}")
    print(f"  Excluded — other (volume/ATR/etc):         {other}")
    print(f"Days scoring >= {threshold}: {above} ({above/len(history)*100:.1f}% of valid days)")


def build_signal_summary_table(
    history, ticker, threshold=6.5,
    extrema_order=10, bottom_tolerance_days=5,
    forward_weeks=(1, 2, 3), verbose=True,
):
    
    from scipy.signal import argrelextrema

    prices = history["price"].values
    dates  = history.index

    minima_idx = argrelextrema(prices, np.less_equal,    order=extrema_order)[0]
    maxima_idx = argrelextrema(prices, np.greater_equal,  order=extrema_order)[0]
    extrema_idx = np.sort(np.concatenate([minima_idx, maxima_idx]))

    # ── Συμπίεση συνεχόμενων signal ημερών σε events ──────────────────────────
    is_signal = (history["score"] >= threshold).fillna(False).values
    event_starts = []
    for i in range(len(is_signal)):
        if is_signal[i] and (i == 0 or not is_signal[i - 1]):
            event_starts.append(i)

    rows = []
    for i in event_starts:
        date        = dates[i]
        signal_price = prices[i]

        # Hit Bottom: υπάρχει local minimum μέσα σε ±tolerance ημέρες;
        nearby_minima = minima_idx[np.abs(minima_idx - i) <= bottom_tolerance_days]
        hit_bottom = "✅" if len(nearby_minima) > 0 else "❌"

        row = {
            "Date":     date.strftime("%d/%m/%Y"),
            "Score":    round(history["score"].iloc[i], 1),
            "RSI":      round(history["rsi"].iloc[i], 0),
            "BB%":      round(history["pct_b"].iloc[i], 2) if pd.notna(history["pct_b"].iloc[i]) else None,
            "Z-Score":  round(history["z_score"].iloc[i], 2) if pd.notna(history["z_score"].iloc[i]) else None,
            "Hit Bottom": hit_bottom,
        }

        # Forward returns σε N εβδομάδες (~5 trading days/εβδομάδα)
        for wk in forward_weeks:
            offset = i + wk * 5
            if offset < len(prices):
                pct = (prices[offset] - signal_price) / signal_price * 100
                row[f"+{wk}w (%)"] = round(pct, 1)
            else:
                row[f"+{wk}w (%)"] = None   # δεν υπάρχει ακόμα αρκετό μέλλον στο dataset

        # Move to Next Extremum: πρώτο local min/max ΜΕΤΑ το signal
        future_extrema = extrema_idx[extrema_idx > i]
        if len(future_extrema) > 0:
            next_idx   = future_extrema[0]
            next_price = prices[next_idx]
            row["Move to Next Extremum (%)"] = round((next_price - signal_price) / signal_price * 100, 1)
        else:
            row["Move to Next Extremum (%)"] = None   # δεν έχει βρεθεί ακόμα (πολύ πρόσφατο signal)

        rows.append(row)

    table = pd.DataFrame(rows)

    if table.empty:
        print(f"Κανένα signal event για {ticker} με threshold >= {threshold}.")
        return table

    hit_rate = (table["Hit Bottom"] == "✅").mean() * 100
    if verbose:
        print(f"{ticker} — {len(table)} signal events (threshold >= {threshold})")
        print(f"Hit rate (κοντά σε πραγματικό local bottom, ±{bottom_tolerance_days}d): {hit_rate:.0f}%")
        for wk in forward_weeks:
            col = f"+{wk}w (%)"
            valid_returns = table[col].dropna()
            if len(valid_returns) > 0:
                print(f"  Μέση απόδοση +{wk}w: {valid_returns.mean():+.1f}%  (n={len(valid_returns)})")

    return table


# ── Weight tuning helpers ───────────────────────────────────────────────────
WEIGHT_KEYS = ["w_rsi", "w_bb", "w_mr", "w_pcr", "w_stochrsi", "w_williams"]


def adjust_weight(weights_cfg, weight_key, new_value):
    
    if weight_key not in WEIGHT_KEYS:
        raise ValueError(f"'{weight_key}' δεν είναι έγκυρο weight key. Επιλογές: {WEIGHT_KEYS}")

    new_cfg = dict(weights_cfg)   # copy — τα periods μένουν ανέγγιχτα
    other_keys = [k for k in WEIGHT_KEYS if k != weight_key and k in new_cfg]
    old_others_sum = sum(new_cfg[k] for k in other_keys)

    if old_others_sum == 0:
        raise ValueError("Δεν μπορεί να γίνει αναδιανομή — τα υπόλοιπα weights αθροίζουν σε 0.")

    scale = (1.0 - new_value) / old_others_sum
    for k in other_keys:
        new_cfg[k] = round(new_cfg[k] * scale, 4)
    new_cfg[weight_key] = new_value

    total = sum(new_cfg[k] for k in WEIGHT_KEYS if k in new_cfg)
    if abs(total - 1.0) > 0.001:
        print(f"⚠️ Προσοχή: τα weights αθροίζουν σε {total:.4f}, όχι 1.0 — έλεγξε.")

    return new_cfg


# ── Batch testing πάνω σε validation set ──────────────────────────────────────
def batch_signal_summary(
    tickers, weights, threshold=6.5, period="10y",
    extrema_order=10, bottom_tolerance_days=5,
    forward_weeks=(1, 2, 3),
):
    
    all_tables = []
    for ticker in tickers:
        try:
            history = compute_daily_score_history(ticker, weights, period=period)
        except Exception as e:
            print(f"⚠️ {ticker}: αποτυχία fetch/compute ({e}) — παραλείπεται")
            continue

        table = build_signal_summary_table(
            history, ticker, threshold,
            extrema_order=extrema_order, bottom_tolerance_days=bottom_tolerance_days,
            forward_weeks=forward_weeks, verbose=False,
        )
        if table is not None and not table.empty:
            table = table.copy()
            table.insert(0, "Ticker", ticker)
            all_tables.append(table)

    if not all_tables:
        print("Κανένα signal event σε κανένα ticker του validation set.")
        return pd.DataFrame()

    combined = pd.concat(all_tables, ignore_index=True)

    hit_rate = (combined["Hit Bottom"] == "✅").mean() * 100
    print(f"{'═' * 60}")
    print(f"BATCH SUMMARY — {len(tickers)} tickers, threshold >= {threshold}")
    print(f"{'═' * 60}")
    print(f"Total signal events: {len(combined)}")
    print(f"Overall hit rate: {hit_rate:.0f}%")
    for wk in forward_weeks:
        col = f"+{wk}w (%)"
        valid = combined[col].dropna()
        if len(valid) > 0:
            print(f"  Μέση απόδοση +{wk}w: {valid.mean():+.1f}%  (n={len(valid)})")
    print(f"{'═' * 60}")

    return combined


def export_score_history(history, table, ticker, threshold=6.5, output_dir="exports"):
    
    from pathlib import Path
    out_path = Path(output_dir).resolve()
    out_path.mkdir(parents=True, exist_ok=True)
    print(f"📁 Αποθήκευση σε: {out_path}")

    # ── Chart -> PDF (vector, καθαρό στην εκτύπωση) ───────────────────────────
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True,
                                     gridspec_kw={"height_ratios": [2, 1]})
    ax1.plot(history.index, history["price"], color="black", linewidth=1)
    signal_days = history[history["score"] >= threshold]
    ax1.scatter(signal_days.index, signal_days["price"], color="green", marker="^",
                s=45, zorder=5, label=f"Score >= {threshold}")
    ax1.set_title(f"{ticker} — Price vs. Daily MR Score")
    ax1.set_ylabel("Price ($)")
    ax1.legend(loc="upper left", fontsize=8)
    ax1.grid(alpha=0.3)
    ax2.plot(history.index, history["score"], color="darkorange", linewidth=1)
    ax2.axhline(threshold, color="green", linestyle="--", alpha=0.6, label=f"threshold={threshold}")
    ax2.set_ylabel("MR Score")
    ax2.set_ylim(0, 10)
    ax2.legend(loc="upper left", fontsize=8)
    ax2.grid(alpha=0.3)
    plt.tight_layout()
    chart_path = out_path / f"{ticker}_chart.pdf"
    fig.savefig(chart_path, bbox_inches="tight")
    plt.close(fig)

    # ── Table -> formatted PDF ────────────────────────────────────────────────
    table_pdf_path = out_path / f"{ticker}_table.pdf"
    table_csv_path = out_path / f"{ticker}_table.csv"
    if table is not None and not table.empty:
        n_rows = len(table)
        fig_h  = max(2, 0.4 * n_rows + 1)
        fig2, ax = plt.subplots(figsize=(11, fig_h))
        ax.axis("off")
        ax.set_title(f"{ticker} — Signal Events (threshold >= {threshold})", fontsize=11, pad=12)
        # matplotlib/PDF export δεν αποδίδει καλά τα colored emoji (✅/❌) — fallback
        # σε άσχετο glyph. Μετατρέπουμε σε plain text ΜΟΝΟ για το PDF table, το
        # DataFrame που επιστρέφεται (και φαίνεται στο notebook) μένει ανέγγιχτο.
        table_for_pdf = table.copy()
        if "Hit Bottom" in table_for_pdf.columns:
            table_for_pdf["Hit Bottom"] = table_for_pdf["Hit Bottom"].map({"✅": "YES", "❌": "NO"})
        mpl_table = ax.table(
            cellText=table_for_pdf.values, colLabels=table_for_pdf.columns,
            cellLoc="center", loc="center",
        )
        mpl_table.auto_set_font_size(False)
        mpl_table.set_fontsize(8)
        mpl_table.scale(1, 1.4)
        plt.tight_layout()
        fig2.savefig(table_pdf_path, bbox_inches="tight")
        plt.close(fig2)
        table.to_csv(table_csv_path, index=False)
        print(f"Saved: {table_pdf_path}")
        print(f"Saved: {table_csv_path}")
    else:
        print("Άδειος πίνακας — δεν αποθηκεύτηκε table PDF/CSV.")

    print(f"Saved: {chart_path}")


In [ ]:
#24 Daily Score History — Single Ticker Run
RUN_DAILY_SCORE_HISTORY = True   # True/False: τρέξε το daily score history παρακάτω

SCORE_HISTORY_TICKER      = "META"
SCORE_HISTORY_THRESHOLD   = 7.5
SCORE_HISTORY_PERIOD      = "10y"    # rolling παράθυρο από ΣΗΜΕΡΑ — άλλαξέ το αν θες άλλο εύρος
SCORE_HISTORY_INTERACTIVE = True     # True = plotly (hover/zoom), False = static matplotlib
SCORE_HISTORY_SHOW_TABLE  = True     # True = εμφάνισε τον πίνακα signal events
SCORE_HISTORY_EXPORT      = False    # True = αποθήκευσε chart+table σε PDF/CSV (φάκελος "exports/")

score_history = None
signal_table  = None
if RUN_DAILY_SCORE_HISTORY:
    score_history = compute_daily_score_history(
        SCORE_HISTORY_TICKER, scorer_weights_cfg, period=SCORE_HISTORY_PERIOD
    )
    if SCORE_HISTORY_INTERACTIVE:
        plot_daily_score_history_interactive(score_history, SCORE_HISTORY_TICKER, SCORE_HISTORY_THRESHOLD)
    else:
        plot_daily_score_history(score_history, SCORE_HISTORY_TICKER, SCORE_HISTORY_THRESHOLD)

    if SCORE_HISTORY_SHOW_TABLE:
        signal_table = build_signal_summary_table(
            score_history, SCORE_HISTORY_TICKER, SCORE_HISTORY_THRESHOLD
        )

    if SCORE_HISTORY_EXPORT:
        export_score_history(
            score_history, signal_table, SCORE_HISTORY_TICKER, SCORE_HISTORY_THRESHOLD
        )

signal_table

## 6. PORTFOLIO TRACKING

In [ ]:
#25 Record Position (demo portfolio) — no manual copy-paste of indicator values
import os
from pathlib import Path

# Points at the demo-portfolio-tracker repo's positions.csv. Override via env
# var if that repo lives somewhere other than a sibling folder locally.
POSITIONS_CSV_PATH = os.environ.get(
    "POSITIONS_CSV_PATH",
    "positions.csv",   # now lives in this same repo, no cross-repo pathing
)


def record_position(ticker, shares, scored, positions_csv_path=POSITIONS_CSV_PATH, entry_date=None):
    """
    Appends a new row to positions.csv, auto-populated from the scorer_mr
    output (`scored`, from compute_scores() a few cells up) for `ticker` —
    every indicator value at entry is captured exactly as scored, no
    manual retyping.
    """
    match = next((r for r in scored if r["ticker"] == ticker), None)
    if match is None:
        raise ValueError(f"{ticker} not found in `scored` — did it pass the scanner's filters today?")

    entry_date = entry_date or pd.Timestamp.today().strftime("%Y-%m-%d")

    row = {
        "ticker":          ticker,
        "entry_date":      entry_date,
        "entry_price":     match["price"],
        "shares":          shares,
        "composite_score": match["composite_score"],
        "setup":           match["setup"],
        "sector":          match["sector"],
        "rsi":             match["rsi"],
        "pct_b":           match["pct_b"],
        "z_score":         match["z_score"],
        "stoch_k":         match["stoch_k"],
        "stoch_d":         match["stoch_d"],
        "williams_r":      match["williams_r"],
        "atr_percentile":  match["atr_percentile"],
        "atr_regime":      match["atr_regime"],
        "vol_ratio":       match["vol_ratio"],
        "pcr_volume":      match["pcr_volume"],
        "pcr_signal":      match["pcr_signal"],
        "avg_volume_m":    match["avg_volume_m"],
        "stop_loss":       match["stop_loss"],
        "target":          match["target_2"],
        "status":          "open",
        "exit_date":       None,
        "exit_price":      None,
    }

    new_row_df = pd.DataFrame([row])
    path = Path(positions_csv_path)
    if path.exists():
        existing = pd.read_csv(path)
        combined = pd.concat([existing, new_row_df], ignore_index=True)
    else:
        combined = new_row_df

    combined.to_csv(path, index=False)
    print(f"Recorded: {ticker} @ ${match['price']:.2f} x {shares} shares (score={match['composite_score']})")
    return row


# ── Call settings ──────────────────────────────────────────────────────────
RECORD_NEW_POSITION = False   # True/False: actually append the row below

POSITION_TICKER = "NCLH"
POSITION_SHARES = 50

if RECORD_NEW_POSITION:
    record_position(POSITION_TICKER, POSITION_SHARES, scored)


In [ ]:
#26 Record Historical Position (for trades already open before you started using this tracker)
"""
Same idea as record_position(), but for a trade you opened in the past —
reconstructs the point-in-time indicator values as of entry_date (using
inspect_signal_quiet from the Batch Signal Inspection cell above — no
look-ahead, exactly what the scanner would have shown that day), while
entry_price is what you ACTUALLY paid, not recomputed from data.

Limitation: pcr_volume/pcr_signal can't be reconstructed retroactively —
no historical options chain was archived before you started the options
scanner. Leave those as None unless you happen to have written the PCR
down yourself at the time (phone tracker notes, etc.) — pass them via
manual_pcr_volume/manual_pcr_signal if so.
"""

def record_historical_position(
    ticker, shares, entry_date, entry_price,
    data, sectors, scorer_weights=None,
    manual_pcr_volume=None, manual_pcr_signal=None,
    manual_stop_loss=None, manual_target=None,
    positions_csv_path=POSITIONS_CSV_PATH,
):
    detail = inspect_signal_quiet(ticker, entry_date, data, sectors, scorer_weights or scorer_weights_cfg)

    if detail is None:
        print(f"⚠️  {ticker} didn't pass hard filters on {entry_date} (or insufficient history that "
              f"far back) — indicator columns will be left blank. The position is still recorded.")

    row = {
        "ticker":          ticker,
        "entry_date":      pd.Timestamp(entry_date).strftime("%Y-%m-%d"),
        "entry_price":     entry_price,   # what you actually paid — not recomputed
        "shares":          shares,
        "composite_score": detail["composite_score"] if detail else None,
        "setup":           detail["setup"] if detail else None,
        "sector":          detail["sector"] if detail else sectors.get(ticker, "Unknown"),
        "rsi":             detail["rsi"] if detail else None,
        "pct_b":           detail["pct_b"] if detail else None,
        "z_score":         detail["z_score"] if detail else None,
        "stoch_k":         detail["stoch_k"] if detail else None,
        "stoch_d":         detail["stoch_d"] if detail else None,
        "williams_r":      detail["williams_r"] if detail else None,
        "atr_percentile":  detail["atr_percentile"] if detail else None,
        "atr_regime":      detail["atr_regime"] if detail else None,
        "vol_ratio":       detail["vol_ratio"] if detail else None,
        "pcr_volume":      manual_pcr_volume,   # not retroactively available — pass manually if you have it
        "pcr_signal":      manual_pcr_signal,
        "avg_volume_m":    detail["avg_volume_m"] if detail else None,
        "stop_loss":       manual_stop_loss,    # your own stop/target from when you opened it, if recorded
        "target":          manual_target,
        "status":          "open",
        "exit_date":       None,
        "exit_price":      None,
    }

    new_row_df = pd.DataFrame([row])
    path = Path(positions_csv_path)
    if path.exists():
        existing = pd.read_csv(path)
        combined = pd.concat([existing, new_row_df], ignore_index=True)
    else:
        combined = new_row_df

    combined.to_csv(path, index=False)
    print(f"Recorded (historical): {ticker} @ ${entry_price:.2f} x {shares} shares, entered {entry_date}")
    return row


# ── Call settings — repeat this block once per already-open position ────────
RECORD_HISTORICAL = False   # True/False: actually append the row below

HIST_TICKER      = "NCLH"
HIST_SHARES      = 50
HIST_ENTRY_DATE  = "2026-06-15"   # the actual date you opened it
HIST_ENTRY_PRICE = 18.50          # what you actually paid

if RECORD_HISTORICAL:
    record_historical_position(
        HIST_TICKER, HIST_SHARES, HIST_ENTRY_DATE, HIST_ENTRY_PRICE,
        data, sectors, scorer_weights_cfg,
    )


In [ ]:
#27 Update Portfolio Tracker (run either script from here — no bash needed)
"""
Two options:
  RUN_PORTFOLIO_BACKFILL       — reconstructs the FULL historical line using
                                  real historical prices from each position's
                                  entry_date to today. Safe to re-run any
                                  time (fully idempotent). Use this while the
                                  daily cron is still off (manual-tuning
                                  phase) — just re-run whenever you want an
                                  updated chart.
  RUN_PORTFOLIO_TRACKER_UPDATE — appends only TODAY's snapshot (what the
                                  daily cron does once enabled). Running
                                  this alone, without ever backfilling,
                                  only ever gives you one point per day you
                                  happen to run it — not a full trajectory.
"""

RUN_PORTFOLIO_BACKFILL       = False   # True/False: run backfill_portfolio_history.py
RUN_PORTFOLIO_TRACKER_UPDATE = False   # True/False: run update_portfolio_tracker.py

if RUN_PORTFOLIO_BACKFILL:
    %run backfill_portfolio_history.py

if RUN_PORTFOLIO_TRACKER_UPDATE:
    %run update_portfolio_tracker.py


In [ ]:
#28 Track Recorded Positions vs VOO
"""
Reads positions.csv (the same file record_position() writes to) and shows,
for each recorded position, its return since entry alongside what the same
holding period would have returned in VOO — the signal values captured at
entry time (composite_score, RSI, z_score, etc.) are shown alongside, as
your original reference point for that trade.

Also plots:
  1. Portfolio % change over time vs benchmark (from portfolio_history.csv,
     the daily automated tracker's output — that file tracks vs SPY, not
     VOO; labeled honestly as SPY below since that's what's actually
     computed there, distinct from the per-position VOO comparison here)
  2. Per-stock alpha bar chart (from the per-position comparison below,
     vs VOO)

This is a read-only view for ad-hoc inspection inside the notebook. The
demo-portfolio-tracker workflow is the authoritative, automated version of
the time-series tracking — this cell just lets you see the same picture
without leaving the notebook.
"""
import yfinance as yf

BENCHMARK_TICKER = "SPY"
PORTFOLIO_HISTORY_PATH = "portfolio_history.csv"

RUN_TRACK_POSITIONS = False   # True/False: run the comparison below
PLOT_PORTFOLIO_VS_BENCHMARK = False   # True/False: time-series line chart
PLOT_PER_STOCK_ALPHA        = False   # True/False: per-position bar chart


def track_positions_vs_benchmark(positions_csv_path=POSITIONS_CSV_PATH, benchmark=BENCHMARK_TICKER):
    path = Path(positions_csv_path)
    if not path.exists():
        print(f"⚠️  {positions_csv_path} not found — no positions recorded yet.")
        return None

    positions = pd.read_csv(path, parse_dates=["entry_date", "exit_date"])
    if positions.empty:
        print("⚠️  positions.csv is empty.")
        return None

    rows = []
    for _, pos in positions.iterrows():
        ticker = pos["ticker"]
        entry_date = pos["entry_date"]
        entry_price = pos["entry_price"]

        if pos["status"] == "closed" and pd.notna(pos.get("exit_price")):
            end_date = pos["exit_date"]
            price_now = pos["exit_price"]
        else:
            end_date = pd.Timestamp.today()
            hist = yf.Ticker(ticker).history(period="5d")
            if hist.empty:
                continue
            price_now = float(hist["Close"].iloc[-1])

        position_return_pct = (price_now - entry_price) / entry_price * 100

        # Same holding period, but in the benchmark
        bench_hist = yf.Ticker(benchmark).history(
            start=entry_date.strftime("%Y-%m-%d"),
            end=(end_date + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
        )
        if bench_hist.empty:
            continue
        bench_entry = float(bench_hist["Close"].iloc[0])
        bench_end   = float(bench_hist["Close"].iloc[-1])
        bench_return_pct = (bench_end - bench_entry) / bench_entry * 100

        alpha_pct = position_return_pct - bench_return_pct

        rows.append({
            "ticker":            ticker,
            "status":            pos["status"],
            "entry_date":        entry_date.date(),
            "entry_price":       round(entry_price, 2),
            "price_now":         round(price_now, 2),   # current price (open) or exit price (closed) — one consistent column
            "return_pct":        round(position_return_pct, 2),
            f"{benchmark}_return_pct": round(bench_return_pct, 2),
            "alpha_pct":         round(alpha_pct, 2),
            "composite_score":   pos.get("composite_score"),
            "rsi":               pos.get("rsi"),
            "z_score":           pos.get("z_score"),
            "pcr_volume":        pos.get("pcr_volume"),
            "sector":            pos.get("sector"),
        })

    result = pd.DataFrame(rows)
    return result


def plot_portfolio_vs_benchmark(history_path=PORTFOLIO_HISTORY_PATH):
    path = Path(history_path)
    if not path.exists():
        print(f"⚠️  {history_path} not found — the daily tracker hasn't run yet.")
        return

    hist = pd.read_csv(path, parse_dates=["date"]).sort_values("date")
    if hist.empty:
        print("⚠️  portfolio_history.csv is empty.")
        return

    if len(hist) == 1:
        print("ℹ️  Only 1 data point so far — the line will just show a single marker. "
              "Run the tracker a few more times (or wait for the daily cron once enabled) to see an actual trend.")

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(hist["date"], hist["portfolio_return_pct"], label="Portfolio", color="#2ecc71", linewidth=2, marker="o", markersize=5)
    ax.plot(hist["date"], hist["spy_return_pct"], label="SPY (from portfolio_history.csv)", color="#95a5a6", linewidth=1.5, linestyle="--", marker="o", markersize=5)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title("Portfolio % Return vs Benchmark, Over Time", fontweight="bold")
    ax.set_ylabel("Return (%)")
    ax.legend(loc="upper left")
    ax.grid(alpha=0.3)
    if len(hist) > 1:
        fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


def plot_per_stock_alpha(comparison_df, benchmark=BENCHMARK_TICKER):
    if comparison_df is None or comparison_df.empty:
        print("⚠️  No comparison data to plot.")
        return

    df = comparison_df.sort_values("alpha_pct")
    colors = ["#e74c3c" if v < 0 else "#2ecc71" for v in df["alpha_pct"]]

    fig, ax = plt.subplots(figsize=(10, max(3, len(df) * 0.4)))
    ax.barh(df["ticker"], df["alpha_pct"], color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"Per-Position Alpha vs {benchmark}", fontweight="bold")
    ax.set_xlabel("Alpha (percentage points)")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()


comparison_df = None
if RUN_TRACK_POSITIONS:
    comparison_df = track_positions_vs_benchmark()
    if comparison_df is not None:
        print(f"{len(comparison_df)} tracked positions vs {BENCHMARK_TICKER}:")
        display(comparison_df)

        avg_alpha = comparison_df["alpha_pct"].mean()
        win_rate = (comparison_df["alpha_pct"] > 0).mean() * 100
        print(f"\nAvg alpha vs {BENCHMARK_TICKER}: {avg_alpha:+.2f}pp  |  Beat benchmark: {win_rate:.0f}% of positions")

if PLOT_PORTFOLIO_VS_BENCHMARK:
    plot_portfolio_vs_benchmark()

if PLOT_PER_STOCK_ALPHA:
    plot_per_stock_alpha(comparison_df)


## 7. WEIGHT EXPERIMENTATION

In [ ]:
#29 Score Diagnostics - Correlation Audit, Weight Sensitivity, Score Distribution

RUN_SCORE_DIAGNOSTICS = False

if RUN_SCORE_DIAGNOSTICS:
    import score_diagnostics as sd

    panel = sd.build_score_panel(
        data, tickers=tickers,
        start="2024-06-01", end="2025-12-31",
        freq="W-TUE",
    )

    sd.correlation_audit(panel)
    sd.weight_sensitivity_analysis(panel, n_samples=200, top_n=TOP_N)
    sd.score_distribution_monitor(panel)

In [ ]:
#30 Weight/Threshold Testing — Baseline (Current Weights)
validation_tickers = ["V", "UNH", "BABA", "MSFT", "GOOGL", "AMZN",
                       "XOM", "NVDA", "ABBV", "NFLX", "PG", "GS","AAPL","CAT"]

RUN_BASELINE=False
if RUN_BASELINE: 
    baseline = batch_signal_summary(validation_tickers, scorer_weights_cfg, threshold=6.5)
    

In [ ]:
#31 Weight/Threshold Testing — Variant (Experiment)
#["w_rsi", "w_bb", "w_mr", "w_pcr", "w_stochrsi", "w_williams"]
test_weights = adjust_weight(scorer_weights_cfg, "w_bb", 0.40)
RUN_VARIANT=False
if RUN_VARIANT: 
    variant = batch_signal_summary(validation_tickers, test_weights, threshold=6.5)

In [ ]:
#32 Out-of-Sample Holdout Test

from out_of_sample import run_oos_test

RUN_OOS_TEST = False

if RUN_OOS_TEST:
    oos_results = run_oos_test(
        backtester_module = backtester,
        data_folder       = DATA_FOLDER,
        confirm           = True,
        data              = data,
        regime_detector   = regime_detector,   # ή None
        top_n             = BACKTEST_TOP_N,
        signal_threshold  = SIGNAL_THRESHOLD,
        exit_score_threshold = EXIT_SCORE_THRESHOLD,
        benchmark_ticker  = BENCHMARK,
        max_hold_days     = MAX_HOLD_DAYS,
        bear_regime_exit  = BEAR_REGIME_EXIT,
        guard_days        = GUARD_DAYS,
        hard_floor_atr    = HARD_FLOOR_ATR,
        trail_trigger_atr = TRAIL_TRIGGER_ATR,
        atr_trail_mult    = ATR_TRAIL_MULT,
        stop_atr_mult     = STOP_ATR_MULT,
        target_atr_mult   = TARGET_ATR_MULT,
        scorer_weights    = scorer_weights_cfg,
        membership        = membership,
    )
    if oos_results:
        backtester.print_backtest_report(oos_results)